# Estimativa de SOH em veículos elétricos com dados reais de recarga

Este notebook implementa um pipeline reproduzível para estimativa do State of Health (SOH) de baterias de veículos elétricos a partir de features extraídas de eventos reais de carregamento.

O pipeline inclui:

- carregamento e auditoria do Dataset Gold por evento;
- remoção de eventos fora da faixa de interesse;
- validação cruzada estratificada com 5 folds;
- balanceamento do conjunto de treino por SMOTER;
- modelo global ExtraTrees;
- três modelos especialistas;
- gate XGBoost para seleção do especialista;
- arquitetura Mixture of Experts;
- roteamento baseado na previsão do modelo global;
- avaliação global e por faixas de SOH;
- geração de predições out-of-fold e tabelas consolidadas.

## Faixa de SOH considerada

Somente eventos com:

\[
0{,}92 \leq SOH \leq 1{,}005
\]

## Arquiteturas avaliadas

1. **ExtraTrees global com mileage**
2. **Mixture of Experts com gate XGBoost**
3. **Mixture of Experts roteado pela previsão global**
4. **Oracle dos especialistas**, utilizado apenas como limite teórico

## Validação

A avaliação é feita por validação cruzada estratificada de 5 folds.  
O SMOTER é aplicado exclusivamente no conjunto de treino de cada fold, evitando vazamento de dados.

# Imports, carregamento, auditoria, limpeza, folds e balanceamento

Esta seção reúne as etapas de preparação do experimento:

- importação das bibliotecas;
- definição das configurações gerais;
- localização portátil das pastas do projeto;
- carregamento do Dataset Gold;
- auditoria e filtragem da faixa de SOH;
- configuração das features;
- criação dos cinco folds externos;
- balanceamento do treino com SMOTER.

Os conjuntos de validação permanecem formados exclusivamente por eventos reais.

In [2]:
# ==========================================================
# 1. IMPORTS E CONFIGURAÇÕES GERAIS
# ==========================================================

from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors

from sklearn.ensemble import ExtraTreesRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)

from xgboost import (
    XGBClassifier,
    XGBRegressor,
)

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Reprodutibilidade
# ----------------------------------------------------------

RANDOM_STATE = 42
N_SPLITS = 5

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ----------------------------------------------------------
# Target e limites
# ----------------------------------------------------------

TARGET_COL = "soh_top10_ref"

SOH_MIN = 0.920
SOH_MAX = 1.005

SOH_BIN_SIZE = 0.005

# ----------------------------------------------------------
# SMOTER
# ----------------------------------------------------------

MILEAGE_SMOTER_MIN = 30_000
MAX_MILEAGE_DISTANCE = 15_000
N_NEIGHBORS_SMOTER = 5

TARGET_FIRST_BIN = 300
TARGET_OTHER_BINS = 500

print("Configuração carregada.")
print("Random state:", RANDOM_STATE)
print("Número de folds:", N_SPLITS)
print("Faixa de SOH:", SOH_MIN, "até", SOH_MAX)

Configuração carregada.
Random state: 42
Número de folds: 5
Faixa de SOH: 0.92 até 1.005


In [3]:
# ==========================================================
# 2. CAMINHOS PORTÁTEIS DO PROJETO
# ==========================================================

from pathlib import Path

# O notebook deve ser executado a partir da pasta NOTEBOOK
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.upper() == "NOTEBOOK":
    PROJECT_ROOT = CURRENT_DIR.parent
elif (CURRENT_DIR / "NOTEBOOK").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto.\n"
        "Abra o notebook dentro da pasta NOTEBOOK ou execute "
        "o Jupyter a partir da raiz SOH_EV_Real_Data."
    )

DATA_DIR = PROJECT_ROOT / "DATA"
CONFIG_DIR = PROJECT_ROOT / "CONFIG"
OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"

AUDIT_DIR = OUTPUT_DIR / "audit"
FOLDS_DIR = OUTPUT_DIR / "folds"
METRICS_DIR = OUTPUT_DIR / "metrics"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
MODELS_DIR = OUTPUT_DIR / "models"

for directory in [
    OUTPUT_DIR,
    AUDIT_DIR,
    FOLDS_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    MODELS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

GOLD_EVENTS_PATH = (
    DATA_DIR
    / "gold_all_events_real.csv"
)

FEATURE_LIST_PATH = (
    DATA_DIR
    / "gold1_column_lists.json"
)

print("=" * 80)
print("CAMINHOS DO PROJETO")
print("=" * 80)

print("Raiz do projeto :", PROJECT_ROOT)
print("Dataset Gold    :", GOLD_EVENTS_PATH)
print("Lista de colunas:", FEATURE_LIST_PATH)
print("Outputs         :", OUTPUT_DIR)

CAMINHOS DO PROJETO
Raiz do projeto : C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data
Dataset Gold    : C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\DATA\gold_all_events_real.csv
Lista de colunas: C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\DATA\gold1_column_lists.json
Outputs         : C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\OUTPUT


In [4]:
# ==========================================================
# 3. CARREGAMENTO DO DATASET GOLD UNIFICADO
# ==========================================================

import json
import numpy as np
import pandas as pd

if not GOLD_EVENTS_PATH.exists():
    raise FileNotFoundError(
        "Dataset Gold não encontrado:\n"
        f"{GOLD_EVENTS_PATH}"
    )

if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(
        "Lista oficial de colunas não encontrada:\n"
        f"{FEATURE_LIST_PATH}"
    )

df_gold = pd.read_csv(
    GOLD_EVENTS_PATH
)

with open(
    FEATURE_LIST_PATH,
    "r",
    encoding="utf-8"
) as file:
    gold_column_lists = json.load(file)

print("=" * 80)
print("DATASET GOLD CARREGADO")
print("=" * 80)

print("Shape:", df_gold.shape)
print("Eventos:", len(df_gold))
print("Carros:", df_gold["car"].nunique())
print("Event IDs únicos:", df_gold["event_id"].nunique())
print("Colunas:", df_gold.shape[1])

DATASET GOLD CARREGADO
Shape: (11660, 346)
Eventos: 11660
Carros: 81
Event IDs únicos: 11660
Colunas: 346


In [5]:
# ==========================================================
# 4. AUDITORIA E LIMPEZA DO DATASET GOLD
# ==========================================================

TARGET_COL = "soh_top10_ref"

SOH_MIN = 0.920
SOH_MAX = 1.005
SOH_BIN_STEP = 0.005

# ----------------------------------------------------------
# 1. Auditorias estruturais
# ----------------------------------------------------------

required_columns = [
    "event_id",
    "car",
    "mileage",
    "charge_segment",
    "sample_origin",
    TARGET_COL,
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_gold.columns
]

if missing_columns:
    raise ValueError(
        f"Colunas obrigatórias ausentes: {missing_columns}"
    )

duplicated_event_ids = int(
    df_gold["event_id"].duplicated().sum()
)

non_real_events = int(
    df_gold["sample_origin"]
    .astype(str)
    .str.lower()
    .ne("real")
    .sum()
)

print("=" * 80)
print("AUDITORIA ESTRUTURAL")
print("=" * 80)

print("Event IDs duplicados:", duplicated_event_ids)
print("Eventos não reais   :", non_real_events)

if duplicated_event_ids > 0:
    raise ValueError(
        "Existem event_id duplicados no Dataset Gold."
    )

if non_real_events > 0:
    raise ValueError(
        "O Dataset Gold inicial deve conter somente eventos reais."
    )

# ----------------------------------------------------------
# 2. Conversão e verificação do target e mileage
# ----------------------------------------------------------

df_gold[TARGET_COL] = pd.to_numeric(
    df_gold[TARGET_COL],
    errors="coerce"
)

df_gold["mileage"] = pd.to_numeric(
    df_gold["mileage"],
    errors="coerce"
)

critical_columns = [
    TARGET_COL,
    "mileage",
]

invalid_critical = (
    df_gold[critical_columns]
    .replace([np.inf, -np.inf], np.nan)
    .isna()
    .sum()
)

print("\nValores inválidos em colunas críticas:")
display(
    invalid_critical.rename("n_invalidos").to_frame()
)

if invalid_critical.sum() > 0:
    raise ValueError(
        "Há NaN ou Inf nas colunas críticas."
    )

# ----------------------------------------------------------
# 3. Filtrar a faixa oficial de SOH
# ----------------------------------------------------------

n_before_filter = len(df_gold)

df_gold_clean = (
    df_gold.loc[
        df_gold[TARGET_COL].between(
            SOH_MIN,
            SOH_MAX,
            inclusive="both"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

n_after_filter = len(df_gold_clean)
n_removed = n_before_filter - n_after_filter

print("\n" + "=" * 80)
print("FILTRO DE SOH")
print("=" * 80)

print("Eventos antes :", n_before_filter)
print("Eventos depois:", n_after_filter)
print("Removidos     :", n_removed)

print(
    "SOH mínimo final:",
    df_gold_clean[TARGET_COL].min()
)

print(
    "SOH máximo final:",
    df_gold_clean[TARGET_COL].max()
)

# ----------------------------------------------------------
# 4. Criar bins fixos de SOH
# ----------------------------------------------------------

SOH_BIN_EDGES = np.round(
    np.arange(
        SOH_MIN,
        SOH_MAX + SOH_BIN_STEP,
        SOH_BIN_STEP
    ),
    6
)

# Garantir que o último limite inclua exatamente 1.005
if SOH_BIN_EDGES[-1] < SOH_MAX:
    SOH_BIN_EDGES = np.append(
        SOH_BIN_EDGES,
        SOH_MAX
    )

df_gold_clean["soh_bin"] = pd.cut(
    df_gold_clean[TARGET_COL],
    bins=SOH_BIN_EDGES,
    include_lowest=True,
    right=True
)

if df_gold_clean["soh_bin"].isna().any():
    n_missing_bins = int(
        df_gold_clean["soh_bin"].isna().sum()
    )

    raise ValueError(
        f"{n_missing_bins} eventos ficaram sem soh_bin."
    )

# ----------------------------------------------------------
# 5. Auditoria geral após limpeza
# ----------------------------------------------------------

audit_clean = pd.DataFrame(
    {
        "metrica": [
            "n_eventos",
            "n_carros",
            "event_id_unicos",
            "soh_min",
            "soh_max",
            "soh_mean",
            "soh_std",
            "mileage_min",
            "mileage_max",
            "mileage_mean",
        ],
        "valor": [
            len(df_gold_clean),
            df_gold_clean["car"].nunique(),
            df_gold_clean["event_id"].nunique(),
            df_gold_clean[TARGET_COL].min(),
            df_gold_clean[TARGET_COL].max(),
            df_gold_clean[TARGET_COL].mean(),
            df_gold_clean[TARGET_COL].std(),
            df_gold_clean["mileage"].min(),
            df_gold_clean["mileage"].max(),
            df_gold_clean["mileage"].mean(),
        ],
    }
)

distribution_by_bin = (
    df_gold_clean
    .groupby(
        "soh_bin",
        observed=False
    )
    .agg(
        n_eventos=("event_id", "size"),
        n_carros=("car", "nunique"),
        soh_min=(TARGET_COL, "min"),
        soh_max=(TARGET_COL, "max"),
        soh_mean=(TARGET_COL, "mean"),
        mileage_mean=("mileage", "mean"),
        mileage_min=("mileage", "min"),
        mileage_max=("mileage", "max"),
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("DATASET GOLD LIMPO")
print("=" * 80)

display(audit_clean)

print("\nDistribuição por bin:")
display(distribution_by_bin)

# ----------------------------------------------------------
# 6. Salvar versão limpa
# ----------------------------------------------------------

CLEAN_GOLD_PATH = (
    DATA_DIR
    / "gold_all_events_real_clean_092_1005.csv"
)

df_gold_clean.to_csv(
    CLEAN_GOLD_PATH,
    index=False
)

audit_clean.to_excel(
    AUDIT_DIR
    / "dataset_clean_audit.xlsx",
    index=False
)

distribution_by_bin.to_excel(
    AUDIT_DIR
    / "dataset_clean_distribution_by_bin.xlsx",
    index=False
)

print("\nArquivo limpo salvo em:")
print(CLEAN_GOLD_PATH)

AUDITORIA ESTRUTURAL
Event IDs duplicados: 0
Eventos não reais   : 0

Valores inválidos em colunas críticas:


,n_invalidos
soh_top10_ref,0
mileage,0



FILTRO DE SOH
Eventos antes : 11660
Eventos depois: 11509
Removidos     : 151
SOH mínimo final: 0.9200012246327732
SOH máximo final: 1.004969147666874

DATASET GOLD LIMPO


,metrica,valor
0,n_eventos,11509.000000
1,n_carros,81.000000
2,event_id_unicos,11509.000000
3,soh_min,0.920001
4,soh_max,1.004969
5,soh_mean,0.979016
6,soh_std,0.016742
7,mileage_min,74.870400
8,mileage_max,102755.664000
9,mileage_mean,43692.357222



Distribuição por bin:


,soh_bin,n_eventos,n_carros,soh_min,soh_max,soh_mean,mileage_mean,mileage_min,mileage_max
0,"(0.919, 0.925]",44,15,0.920001,0.924860,0.922596,59935.968000,13499.3760,90454.5312
1,"(0.925, 0.93]",92,26,0.925004,0.929947,0.927633,66046.064557,13028.2944,89614.9056
2,"(0.93, 0.935]",118,28,0.930019,0.934993,0.932445,62794.552271,9023.4144,89984.0832
3,"(0.935, 0.94]",146,36,0.935005,0.939922,0.937416,64109.916953,11623.0752,98064.5952
4,"(0.94, 0.945]",225,45,0.940034,0.944989,0.942760,60960.492032,9932.1024,97275.7632
5,"(0.945, 0.95]",250,52,0.945013,0.949934,0.947521,57898.000934,10545.3216,93389.0496
6,"(0.95, 0.955]",328,55,0.950006,0.954987,0.952356,53573.944332,10377.5232,92075.2800
7,"(0.955, 0.96]",406,63,0.955013,0.960000,0.957768,54258.426514,6084.9888,101916.7776
8,"(0.96, 0.965]",543,70,0.960001,0.965000,0.962521,54254.138643,2240.9376,101073.5616
9,"(0.965, 0.97]",664,75,0.965009,0.969981,0.967566,50337.867933,623.9904,101711.4912



Arquivo limpo salvo em:
C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\DATA\gold_all_events_real_clean_092_1005.csv


## 5. Configuração das features dos modelos finais

As listas de features utilizadas neste estudo foram previamente selecionadas durante a etapa de desenvolvimento dos modelos.

Para evitar viés de seleção durante a validação cruzada, essas listas são mantidas fixas em todos os folds.

### Modelo global e especialistas

São utilizadas as 20 features físicas campeãs da regressão, acrescidas de `mileage`, totalizando 21 entradas.

### Gate do Mixture of Experts

O gate utiliza as 75 features selecionadas pelo XGBoost e posteriormente limpas pela remoção de:

- features dependentes da faixa `SOC_0_20`;
- `charge_profile_current_code_v1`.

A variável `mileage` é adicionada ao gate, totalizando 76 entradas.

Nenhuma seleção adicional de features é realizada durante a validação cruzada.

In [6]:
# ==========================================================
# 5. CARREGAR A LISTA DAS FEATURES DO GATE
#    DIRETAMENTE DO NOVO PROJETO
# ==========================================================

import json
import numpy as np
import pandas as pd

GATE_FEATURE_AUDIT_PATH = (
    CONFIG_DIR
    / "gate_80_features_audit.xlsx"
)

if not GATE_FEATURE_AUDIT_PATH.exists():
    raise FileNotFoundError(
        "Arquivo de auditoria das features do gate "
        "não encontrado:\n"
        f"{GATE_FEATURE_AUDIT_PATH}\n\n"
        "Confirme que o arquivo está dentro da pasta CONFIG."
    )

gate_features_audit = pd.read_excel(
    GATE_FEATURE_AUDIT_PATH
)

required_audit_columns = [
    "feature",
    "status",
]

missing_audit_columns = [
    column
    for column in required_audit_columns
    if column not in gate_features_audit.columns
]

if missing_audit_columns:
    raise ValueError(
        "O arquivo de auditoria não possui as colunas "
        f"esperadas: {missing_audit_columns}"
    )

gate_features_audit["feature"] = (
    gate_features_audit["feature"]
    .astype(str)
    .str.strip()
)

gate_features_audit["status"] = (
    gate_features_audit["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

FEATURES_GATE_80_ORIGINAL = (
    gate_features_audit["feature"]
    .tolist()
)

FEATURES_GATE_REMOVED = (
    gate_features_audit.loc[
        gate_features_audit["status"].eq("REMOVER"),
        "feature",
    ]
    .tolist()
)

FEATURES_GATE_75 = (
    gate_features_audit.loc[
        gate_features_audit["status"].eq("MANTER"),
        "feature",
    ]
    .tolist()
)

print("=" * 90)
print("FEATURES DO GATE CARREGADAS")
print("=" * 90)

print("Arquivo:")
print(GATE_FEATURE_AUDIT_PATH)

print(
    "\nFeatures originais:",
    len(FEATURES_GATE_80_ORIGINAL)
)

print(
    "Features removidas:",
    len(FEATURES_GATE_REMOVED)
)

print(
    "Features finais:",
    len(FEATURES_GATE_75)
)

print("\nFeatures removidas:")

for feature in FEATURES_GATE_REMOVED:
    print("-", feature)

if len(FEATURES_GATE_80_ORIGINAL) != 80:
    raise ValueError(
        "A lista original deveria possuir 80 features, "
        f"mas possui {len(FEATURES_GATE_80_ORIGINAL)}."
    )

if len(FEATURES_GATE_REMOVED) != 5:
    raise ValueError(
        "Deveriam existir exatamente cinco features "
        f"removidas, mas existem {len(FEATURES_GATE_REMOVED)}."
    )

if len(FEATURES_GATE_75) != 75:
    raise ValueError(
        "A lista final deveria possuir 75 features, "
        f"mas possui {len(FEATURES_GATE_75)}."
    )

print("\nLista das 75 features carregada com sucesso.")

FEATURES DO GATE CARREGADAS
Arquivo:
C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\CONFIG\gate_80_features_audit.xlsx

Features originais: 80
Features removidas: 5
Features finais: 75

Features removidas:
- charge_profile_current_code_v1
- temp_mean_SOC_0_20_p95
- volt_SOC_0_20_max
- volt_SOC_0_20_std
- volt_SOC_0_20_mean

Lista das 75 features carregada com sucesso.


In [10]:
# ==========================================================
# 6. DEFINIÇÃO DAS FEATURES FINAIS DOS MODELOS
# ==========================================================

import json
import numpy as np
import pandas as pd

# ----------------------------------------------------------
# 1. Features físicas campeãs da regressão
# ----------------------------------------------------------

FEATURES_REGRESSION_20 = [
    "min_single_volt_slope",
    "min_single_volt_iqr",
    "volt_slope",
    "temp_mean_mean",
    "cell_imbalance_SOC_40_60_std",
    "cell_imbalance_SOC_40_60_p95",
    "cell_imbalance_SOC_80_100_std",
    "power_proxy_SOC_60_80_std",
    "temp_mean_SOC_40_60_mean",
    "min_temp_median",
    "min_temp_p05",
    "current_abs_p05",
    "cell_imbalance_SOC_60_80_std",
    "current_p05",
    "current_abs_SOC_60_80_mean",
    "current_abs_SOC_40_60_mean",
    "cell_imbalance_SOC_60_80_max",
    "power_proxy_median",
    "power_proxy_p05",
    "cell_imbalance_p75",
]

# ----------------------------------------------------------
# 2. Listas finais com mileage
# ----------------------------------------------------------

FEATURES_GLOBAL = list(
    dict.fromkeys(
        FEATURES_REGRESSION_20
        + ["mileage"]
    )
)

FEATURES_EXPERTS = (
    FEATURES_GLOBAL.copy()
)

FEATURES_GATE = list(
    dict.fromkeys(
        FEATURES_GATE_75
        + ["mileage"]
    )
)

# ----------------------------------------------------------
# 3. Configuração dos especialistas
# ----------------------------------------------------------

EXPERT_CONFIG = {
    0: {
        "name": "A_XGBoost_Baixo",
        "model": "XGBRegressor",
        "soh_min": 0.920,
        "soh_max": 0.945,
    },
    1: {
        "name": "B_ExtraTrees_Intermediario",
        "model": "ExtraTreesRegressor",
        "soh_min": 0.940,
        "soh_max": 0.975,
    },
    2: {
        "name": "C_ExtraTrees_Alto",
        "model": "ExtraTreesRegressor",
        "soh_min": 0.970,
        "soh_max": 1.005,
    },
}

# ----------------------------------------------------------
# 4. Auditoria das listas
# ----------------------------------------------------------

feature_sets = {
    "Global": FEATURES_GLOBAL,
    "Especialistas": FEATURES_EXPERTS,
    "Gate": FEATURES_GATE,
}

audit_rows = []

for set_name, feature_list in feature_sets.items():

    missing_features = [
        feature
        for feature in feature_list
        if feature not in df_gold_clean.columns
    ]

    duplicated_features = (
        pd.Index(feature_list)[
            pd.Index(feature_list).duplicated()
        ]
        .tolist()
    )

    if missing_features:
        n_invalid = np.nan
    else:
        n_invalid = int(
            df_gold_clean[feature_list]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .isna()
            .sum()
            .sum()
        )

    audit_rows.append(
        {
            "conjunto": set_name,
            "n_features": len(feature_list),
            "n_duplicadas": len(
                duplicated_features
            ),
            "n_ausentes": len(
                missing_features
            ),
            "n_nan_inf": n_invalid,
            "duplicadas": duplicated_features,
            "ausentes": missing_features,
        }
    )

feature_sets_audit = pd.DataFrame(
    audit_rows
)

print("=" * 90)
print("FEATURES FINAIS DOS MODELOS")
print("=" * 90)

print(
    "Features físicas da regressão:",
    len(FEATURES_REGRESSION_20)
)

print(
    "Global com mileage:",
    len(FEATURES_GLOBAL)
)

print(
    "Especialistas com mileage:",
    len(FEATURES_EXPERTS)
)

print(
    "Gate sem mileage:",
    len(FEATURES_GATE_75)
)

print(
    "Gate com mileage:",
    len(FEATURES_GATE)
)

display(feature_sets_audit)

# ----------------------------------------------------------
# 5. Validações obrigatórias
# ----------------------------------------------------------

if len(FEATURES_REGRESSION_20) != 20:
    raise ValueError(
        "A regressão deveria possuir 20 features físicas."
    )

if len(FEATURES_GLOBAL) != 21:
    raise ValueError(
        "O modelo global deveria possuir "
        "20 features físicas + mileage."
    )

if len(FEATURES_EXPERTS) != 21:
    raise ValueError(
        "Os especialistas deveriam possuir "
        "20 features físicas + mileage."
    )

if len(FEATURES_GATE_75) != 75:
    raise ValueError(
        "O gate deveria possuir 75 features limpas."
    )

if len(FEATURES_GATE) != 76:
    raise ValueError(
        "O gate deveria possuir "
        "75 features limpas + mileage."
    )

if feature_sets_audit["n_ausentes"].sum() > 0:
    raise ValueError(
        "Existem features ausentes no Dataset Gold."
    )

if feature_sets_audit["n_duplicadas"].sum() > 0:
    raise ValueError(
        "Existem features duplicadas nas listas."
    )

if feature_sets_audit["n_nan_inf"].sum() > 0:
    raise ValueError(
        "Existem NaN ou Inf nas features finais."
    )

# ----------------------------------------------------------
# 6. Salvar configuração portátil
# ----------------------------------------------------------

FINAL_FEATURE_CONFIG_PATH = (
    CONFIG_DIR
    / "final_feature_lists.json"
)

final_feature_config = {
    "target": TARGET_COL,
    "regression_physical_features_20":
        FEATURES_REGRESSION_20,
    "global_features_with_mileage":
        FEATURES_GLOBAL,
    "expert_features_with_mileage":
        FEATURES_EXPERTS,
    "gate_original_features_80":
        FEATURES_GATE_80_ORIGINAL,
    "gate_removed_features_5":
        FEATURES_GATE_REMOVED,
    "gate_clean_features_75":
        FEATURES_GATE_75,
    "gate_features_with_mileage":
        FEATURES_GATE,
    "expert_configuration":
        EXPERT_CONFIG,
}

with open(
    FINAL_FEATURE_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_feature_config,
        file,
        ensure_ascii=False,
        indent=4,
    )

feature_sets_audit.to_excel(
    AUDIT_DIR
    / "feature_sets_audit.xlsx",
    index=False,
)

print("\nConfiguração salva em:")
print(FINAL_FEATURE_CONFIG_PATH)

FEATURES FINAIS DOS MODELOS
Features físicas da regressão: 20
Global com mileage: 21
Especialistas com mileage: 21
Gate sem mileage: 75
Gate com mileage: 76


,conjunto,n_features,n_duplicadas,n_ausentes,n_nan_inf,duplicadas,ausentes
0,Global,21,0,0,0,[],[]
1,Especialistas,21,0,0,0,[],[]
2,Gate,76,0,0,0,[],[]



Configuração salva em:
C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\CONFIG\final_feature_lists.json


## 6. Criação dos folds de validação cruzada

Os cinco folds são construídos por evento, utilizando estratificação pelos bins de SOH de largura igual a 0,005.

Nesta configuração:

- cada evento participa da validação exatamente uma vez;
- os demais quatro folds compõem o conjunto de treino;
- os conjuntos de validação contêm exclusivamente eventos reais;
- o SMOTER será aplicado posteriormente e somente ao treino de cada fold;
- os mesmos veículos podem aparecer no treino e na validação, reproduzindo o cenário intra-frota adotado durante o desenvolvimento dos modelos.

A atribuição dos folds é mantida fixa por meio de `RANDOM_STATE`.

In [11]:
# ==========================================================
# 6. CRIAÇÃO DOS 5 FOLDS ESTRATIFICADOS
# ==========================================================

from sklearn.model_selection import StratifiedKFold

# ----------------------------------------------------------
# 1. Preparar o dataset-base
# ----------------------------------------------------------

df_cv = (
    df_gold_clean
    .copy()
    .reset_index(drop=True)
)

if "soh_bin" not in df_cv.columns:
    raise KeyError(
        "A coluna 'soh_bin' não foi encontrada. "
        "Execute primeiro a célula de limpeza."
    )

if df_cv["soh_bin"].isna().any():
    raise ValueError(
        "Existem eventos sem soh_bin."
    )

# Rótulo usado para estratificação
stratification_labels = (
    df_cv["soh_bin"]
    .astype(str)
    .to_numpy()
)

# ----------------------------------------------------------
# 2. Verificar se todos os bins suportam cinco folds
# ----------------------------------------------------------

bin_counts_before_cv = (
    df_cv["soh_bin"]
    .value_counts(
        sort=False,
        dropna=False
    )
    .rename("n_eventos")
    .reset_index()
    .rename(
        columns={
            "index": "soh_bin"
        }
    )
)

bins_with_fewer_than_five = (
    bin_counts_before_cv.loc[
        bin_counts_before_cv["n_eventos"]
        < N_SPLITS
    ]
    .copy()
)

print("=" * 90)
print("AUDITORIA PRÉ-VALIDAÇÃO CRUZADA")
print("=" * 90)

print("Eventos:", len(df_cv))
print("Carros :", df_cv["car"].nunique())
print("Folds  :", N_SPLITS)

print("\nDistribuição geral por bin:")
display(bin_counts_before_cv)

if len(bins_with_fewer_than_five) > 0:

    print(
        "\nATENÇÃO: existem bins com menos eventos "
        "que o número de folds:"
    )

    display(bins_with_fewer_than_five)

    raise ValueError(
        "Não é possível aplicar StratifiedKFold com "
        f"{N_SPLITS} folds enquanto houver bins com menos "
        f"de {N_SPLITS} eventos."
    )

# ----------------------------------------------------------
# 3. Criar os folds
# ----------------------------------------------------------

cv_splitter = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

df_cv["cv_fold"] = -1

for fold_id, (_, validation_indices) in enumerate(
    cv_splitter.split(
        X=np.zeros(
            len(df_cv),
            dtype=np.uint8
        ),
        y=stratification_labels,
    )
):

    df_cv.loc[
        validation_indices,
        "cv_fold"
    ] = fold_id

df_cv["cv_fold"] = (
    df_cv["cv_fold"]
    .astype(int)
)

# ----------------------------------------------------------
# 4. Validações estruturais
# ----------------------------------------------------------

if df_cv["cv_fold"].lt(0).any():
    raise ValueError(
        "Existem eventos sem fold atribuído."
    )

expected_folds = set(
    range(N_SPLITS)
)

generated_folds = set(
    df_cv["cv_fold"].unique().tolist()
)

if generated_folds != expected_folds:
    raise ValueError(
        f"Os folds gerados foram {generated_folds}, "
        f"mas o esperado era {expected_folds}."
    )

if df_cv["event_id"].duplicated().any():
    raise ValueError(
        "Existem event_id duplicados após a criação "
        "dos folds."
    )

# Cada evento deve estar associado a apenas um fold
events_per_fold_assignment = (
    df_cv
    .groupby("event_id")["cv_fold"]
    .nunique()
)

if not events_per_fold_assignment.eq(1).all():
    raise ValueError(
        "Algum evento foi atribuído a mais de um fold."
    )

# ----------------------------------------------------------
# 5. Resumo geral dos folds
# ----------------------------------------------------------

fold_summary = (
    df_cv
    .groupby("cv_fold")
    .agg(
        n_eventos=("event_id", "size"),
        n_carros=("car", "nunique"),
        soh_min=(TARGET_COL, "min"),
        soh_max=(TARGET_COL, "max"),
        soh_mean=(TARGET_COL, "mean"),
        soh_std=(TARGET_COL, "std"),
        mileage_min=("mileage", "min"),
        mileage_max=("mileage", "max"),
        mileage_mean=("mileage", "mean"),
    )
    .reset_index()
)

fold_summary["percentual_eventos_%"] = (
    100
    * fold_summary["n_eventos"]
    / len(df_cv)
).round(2)

# ----------------------------------------------------------
# 6. Distribuição dos bins em cada fold
# ----------------------------------------------------------

fold_bin_distribution = pd.crosstab(
    index=df_cv["soh_bin"],
    columns=df_cv["cv_fold"],
    margins=True,
)

fold_bin_distribution.index.name = (
    "soh_bin"
)

fold_bin_distribution.columns.name = (
    "cv_fold"
)

fold_bin_distribution_long = (
    df_cv
    .groupby(
        [
            "cv_fold",
            "soh_bin",
        ],
        observed=False,
    )
    .agg(
        n_eventos=("event_id", "size"),
        n_carros=("car", "nunique"),
        soh_mean=(TARGET_COL, "mean"),
        mileage_mean=("mileage", "mean"),
    )
    .reset_index()
)

# ----------------------------------------------------------
# 7. Auditoria treino × validação
# ----------------------------------------------------------

fold_train_valid_rows = []

for fold_id in range(N_SPLITS):

    validation_mask = (
        df_cv["cv_fold"].eq(fold_id)
    )

    train_fold = (
        df_cv.loc[
            ~validation_mask
        ]
        .copy()
    )

    validation_fold = (
        df_cv.loc[
            validation_mask
        ]
        .copy()
    )

    shared_event_ids = len(
        set(
            train_fold["event_id"]
        ).intersection(
            set(
                validation_fold["event_id"]
            )
        )
    )

    shared_cars = len(
        set(
            train_fold["car"]
        ).intersection(
            set(
                validation_fold["car"]
            )
        )
    )

    fold_train_valid_rows.append(
        {
            "fold": fold_id,

            "n_train_real":
                len(train_fold),

            "n_valid_real":
                len(validation_fold),

            "cars_train":
                train_fold["car"].nunique(),

            "cars_valid":
                validation_fold["car"].nunique(),

            "cars_shared":
                shared_cars,

            "event_ids_shared":
                shared_event_ids,

            "train_soh_min":
                train_fold[TARGET_COL].min(),

            "train_soh_max":
                train_fold[TARGET_COL].max(),

            "valid_soh_min":
                validation_fold[TARGET_COL].min(),

            "valid_soh_max":
                validation_fold[TARGET_COL].max(),

            "train_soh_mean":
                train_fold[TARGET_COL].mean(),

            "valid_soh_mean":
                validation_fold[TARGET_COL].mean(),

            "train_mileage_mean":
                train_fold["mileage"].mean(),

            "valid_mileage_mean":
                validation_fold["mileage"].mean(),
        }
    )

fold_train_valid_audit = pd.DataFrame(
    fold_train_valid_rows
)

if (
    fold_train_valid_audit[
        "event_ids_shared"
    ].sum()
    > 0
):
    raise ValueError(
        "Foi detectado vazamento de event_id entre "
        "treino e validação."
    )

# ----------------------------------------------------------
# 8. Conferência do número total de validações
# ----------------------------------------------------------

total_validation_events = int(
    fold_summary["n_eventos"].sum()
)

if total_validation_events != len(df_cv):
    raise ValueError(
        "A soma dos eventos de validação dos folds "
        "não coincide com o dataset."
    )

# ----------------------------------------------------------
# 9. Exibir os resultados
# ----------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMO DOS 5 FOLDS")
print("=" * 90)

display(fold_summary)

print("\nAuditoria treino × validação:")
display(fold_train_valid_audit)

print("\nDistribuição de eventos por bin e fold:")
display(fold_bin_distribution)

# ----------------------------------------------------------
# 10. Salvar as atribuições e auditorias
# ----------------------------------------------------------

FOLD_ASSIGNMENT_PATH = (
    DATA_DIR
    / "gold_clean_5fold_assignments.csv"
)

df_cv.to_csv(
    FOLD_ASSIGNMENT_PATH,
    index=False
)

fold_summary.to_excel(
    AUDIT_DIR
    / "cv_fold_summary.xlsx",
    index=False
)

fold_train_valid_audit.to_excel(
    AUDIT_DIR
    / "cv_train_validation_audit.xlsx",
    index=False
)

fold_bin_distribution.to_excel(
    AUDIT_DIR
    / "cv_bin_distribution_wide.xlsx"
)

fold_bin_distribution_long.to_excel(
    AUDIT_DIR
    / "cv_bin_distribution_long.xlsx",
    index=False
)

print("\n" + "=" * 90)
print("FOLDS CRIADOS COM SUCESSO")
print("=" * 90)

print("Dataset com atribuição dos folds:")
print(FOLD_ASSIGNMENT_PATH)

AUDITORIA PRÉ-VALIDAÇÃO CRUZADA
Eventos: 11509
Carros : 81
Folds  : 5

Distribuição geral por bin:


,soh_bin,n_eventos
0,"(0.919, 0.925]",44
1,"(0.925, 0.93]",92
2,"(0.93, 0.935]",118
3,"(0.935, 0.94]",146
4,"(0.94, 0.945]",225
5,"(0.945, 0.95]",250
6,"(0.95, 0.955]",328
7,"(0.955, 0.96]",406
8,"(0.96, 0.965]",543
9,"(0.965, 0.97]",664



RESUMO DOS 5 FOLDS


,cv_fold,n_eventos,n_carros,soh_min,soh_max,soh_mean,soh_std,mileage_min,mileage_max,mileage_mean,percentual_eventos_%
0,0,2302,81,0.920840,1.004969,0.979003,0.016692,74.8704,101916.7776,43782.696776,20.00
1,1,2302,81,0.920233,1.004848,0.979057,0.016760,376.6752,101319.5040,44080.811906,20.00
2,2,2302,81,0.920001,1.004355,0.978980,0.016803,342.9888,101073.5616,43544.386568,20.00
3,3,2302,81,0.921300,1.004355,0.979008,0.016735,235.5936,102120.5856,43533.279599,20.00
4,4,2301,81,0.920436,1.004822,0.979032,0.016733,623.9904,102755.6640,43520.536624,19.99



Auditoria treino × validação:


,fold,n_train_real,n_valid_real,cars_train,cars_valid,cars_shared,event_ids_shared,train_soh_min,train_soh_max,valid_soh_min,valid_soh_max,train_soh_mean,valid_soh_mean,train_mileage_mean,valid_mileage_mean
0,0,9207,2302,81,81,81,0,0.920001,1.004848,0.920840,1.004969,0.979019,0.979003,43669.769881,43782.696776
1,1,9207,2302,81,81,81,0,0.920001,1.004969,0.920233,1.004848,0.979006,0.979057,43595.233004,44080.811906
2,2,9207,2302,81,81,81,0,0.920233,1.004969,0.920001,1.004355,0.979025,0.978980,43729.353904,43544.386568
3,3,9207,2302,81,81,81,0,0.920001,1.004969,0.921300,1.004355,0.979018,0.979008,43732.130948,43533.279599
4,4,9208,2301,81,81,81,0,0.920001,1.004969,0.920436,1.004822,0.979012,0.979032,43735.293712,43520.536624



Distribuição de eventos por bin e fold:


cv_fold,0,1,2,3,4,All
soh_bin,,,,,,
"(0.919, 0.925]",9,9,9,9,8,44
"(0.925, 0.93]",19,18,18,18,19,92
"(0.93, 0.935]",23,24,24,24,23,118
"(0.935, 0.94]",29,29,29,29,30,146
"(0.94, 0.945]",45,45,45,45,45,225
"(0.945, 0.95]",50,50,50,50,50,250
"(0.95, 0.955]",65,66,66,66,65,328
"(0.955, 0.96]",82,81,81,81,81,406
"(0.96, 0.965]",108,108,109,109,109,543



FOLDS CRIADOS COM SUCESSO
Dataset com atribuição dos folds:
C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\DATA\gold_clean_5fold_assignments.csv


## 7. Balanceamento do treino por SMOTER condicionado

O SMOTER é aplicado separadamente ao conjunto de treino de cada fold.

Os eventos da validação nunca participam:

- do ajuste do `RobustScaler`;
- da busca de vizinhos;
- da geração de eventos sintéticos;
- da definição dos valores interpolados.

### Regras adotadas

- apenas eventos com `mileage ≥ 30.000 km` podem ser pais;
- os dois pais devem pertencer ao mesmo bin de SOH;
- a diferença de mileage entre os pais deve ser menor ou igual a 15.000 km;
- a distância entre eventos é calculada utilizando as 250 features físicas oficiais normalizadas com `RobustScaler`;
- são considerados no máximo cinco vizinhos por evento;
- o bin entre 0,920 e 0,925 é elevado a 300 eventos;
- os demais bins com menos de 500 eventos são elevados a 500;
- nenhum evento real é removido;
- os conjuntos de validação permanecem exclusivamente reais.

In [12]:
# ==========================================================
# 7. FUNÇÃO DE SMOTER CONDICIONADO POR FOLD
# ==========================================================

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import pairwise_distances

# ----------------------------------------------------------
# 1. Recuperar as 250 features físicas oficiais
# ----------------------------------------------------------

if "MODEL_FEATURES_GOLD1_FINAL" not in gold_column_lists:
    raise KeyError(
        "A chave 'MODEL_FEATURES_GOLD1_FINAL' não foi "
        "encontrada em gold1_column_lists.json."
    )

FEATURE_COLS = list(
    gold_column_lists[
        "MODEL_FEATURES_GOLD1_FINAL"
    ]
)

print("=" * 90)
print("CONFIGURAÇÃO DO SMOTER")
print("=" * 90)

print("Features físicas oficiais:", len(FEATURE_COLS))

if len(FEATURE_COLS) != 250:
    raise ValueError(
        "Eram esperadas 250 features físicas, "
        f"mas foram encontradas {len(FEATURE_COLS)}."
    )

missing_smoter_features = [
    feature
    for feature in FEATURE_COLS
    if feature not in df_cv.columns
]

if missing_smoter_features:
    raise ValueError(
        "Existem features oficiais ausentes no dataset:\n"
        f"{missing_smoter_features}"
    )

# ----------------------------------------------------------
# 2. Configuração oficial do SMOTER
# ----------------------------------------------------------

MILEAGE_SMOTER_MIN = 30_000
MAX_MILEAGE_DIFF = 15_000
K_NEIGHBORS_SMOTER = 5

TARGET_FIRST_BIN = 300
TARGET_OTHER_BINS = 500

SMOTER_RANDOM_STATE = RANDOM_STATE

# ----------------------------------------------------------
# 3. Função de definição do alvo de cada bin
# ----------------------------------------------------------

def definir_alvo_smoter(intervalo):
    """
    Define o número desejado de eventos no treino final.

    Regras:
    - primeiro bin de SOH: alvo de 300 eventos;
    - demais bins até 1.005: alvo de 500 eventos;
    - bins que já possuem quantidade maior ou igual ao alvo
      permanecem inalterados.
    """

    left = float(intervalo.left)
    right = float(intervalo.right)

    tolerance = 1e-8

    # Primeiro bin criado com include_lowest=True pode aparecer
    # com limite esquerdo ligeiramente inferior a 0.920.
    if (
        right <= 0.925 + tolerance
        and right > 0.920
    ):
        return TARGET_FIRST_BIN

    if (
        right > 0.925 + tolerance
        and right <= SOH_MAX + tolerance
    ):
        return TARGET_OTHER_BINS

    return None


# ----------------------------------------------------------
# 4. Função principal de balanceamento
# ----------------------------------------------------------

def aplicar_smoter_fold(
    train_real_fold,
    fold_id,
    feature_cols=FEATURE_COLS,
    random_state=SMOTER_RANDOM_STATE,
):
    """
    Aplica o SMOTER somente ao treino real de um fold.

    Retorna
    -------
    train_balanced_fold : DataFrame
        Eventos reais e sintéticos do treino.

    synthetic_fold : DataFrame
        Apenas os eventos sintéticos.

    balance_plan_fold : DataFrame
        Plano de balanceamento por bin.

    neighbor_audit_fold : DataFrame
        Auditoria da disponibilidade de vizinhos.

    generation_audit_fold : DataFrame
        Identificação dos pais e parâmetros de interpolação.
    """

    rng = np.random.default_rng(
        random_state + int(fold_id)
    )

    train_real_fold = (
        train_real_fold
        .copy()
        .reset_index(drop=True)
    )

    train_real_fold["sample_origin"] = "real"
    train_real_fold["split"] = "train_real"
    train_real_fold["source_fold"] = int(fold_id)

    # ------------------------------------------------------
    # 4.1. Verificações iniciais
    # ------------------------------------------------------

    if train_real_fold["event_id"].duplicated().any():
        raise ValueError(
            f"Fold {fold_id}: existem event_id duplicados "
            "no treino real."
        )

    missing_features = [
        feature
        for feature in feature_cols
        if feature not in train_real_fold.columns
    ]

    if missing_features:
        raise ValueError(
            f"Fold {fold_id}: features ausentes:\n"
            f"{missing_features}"
        )

    # ------------------------------------------------------
    # 4.2. Eventos elegíveis para gerar sintéticos
    # ------------------------------------------------------

    eligible = (
        train_real_fold.loc[
            train_real_fold["mileage"]
            >= MILEAGE_SMOTER_MIN
        ]
        .copy()
        .reset_index()
        .rename(
            columns={
                "index": "train_real_index"
            }
        )
    )

    if len(eligible) == 0:
        raise ValueError(
            f"Fold {fold_id}: nenhum evento elegível "
            "para o SMOTER."
        )

    # ------------------------------------------------------
    # 4.3. Preparação numérica e RobustScaler
    # ------------------------------------------------------

    X_eligible = (
        eligible[feature_cols]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    feature_medians = X_eligible.median()

    X_eligible_imputed = (
        X_eligible.fillna(
            feature_medians
        )
    )

    remaining_invalid = int(
        X_eligible_imputed
        .isna()
        .sum()
        .sum()
    )

    if remaining_invalid > 0:
        raise ValueError(
            f"Fold {fold_id}: ainda existem "
            f"{remaining_invalid} valores inválidos "
            "após a imputação."
        )

    feature_nunique = (
        X_eligible_imputed
        .nunique(dropna=False)
    )

    constant_features = (
        feature_nunique.loc[
            feature_nunique <= 1
        ]
        .index
        .tolist()
    )

    smoter_distance_features = [
        feature
        for feature in feature_cols
        if feature not in constant_features
    ]

    if len(smoter_distance_features) == 0:
        raise ValueError(
            f"Fold {fold_id}: nenhuma feature variável "
            "disponível para calcular distâncias."
        )

    scaler = RobustScaler()

    X_scaled = scaler.fit_transform(
        X_eligible_imputed[
            smoter_distance_features
        ]
    )

    X_scaled_df = pd.DataFrame(
        X_scaled,
        columns=smoter_distance_features,
        index=eligible.index,
    )

    # Matriz original usada na interpolação
    X_original = (
        X_eligible_imputed[
            feature_cols
        ]
        .reset_index(drop=True)
    )

    # ------------------------------------------------------
    # 4.4. Busca de vizinhos condicionados
    # ------------------------------------------------------

    neighbor_map = {}
    neighbor_audit_rows = []

    for soh_bin_value, bin_df in eligible.groupby(
        "soh_bin",
        observed=True,
    ):

        bin_indices = (
            bin_df.index.to_numpy()
        )

        n_bin = len(bin_indices)

        if n_bin < 2:

            for local_idx in bin_indices:

                neighbor_map[int(local_idx)] = []

                neighbor_audit_rows.append(
                    {
                        "fold": int(fold_id),
                        "local_index": int(local_idx),
                        "event_id": eligible.loc[
                            local_idx,
                            "event_id",
                        ],
                        "soh_bin": str(
                            soh_bin_value
                        ),
                        "mileage": float(
                            eligible.loc[
                                local_idx,
                                "mileage",
                            ]
                        ),
                        "n_candidates_mileage": 0,
                        "n_neighbors_available": 0,
                    }
                )

            continue

        X_bin = (
            X_scaled_df.loc[
                bin_indices
            ]
            .to_numpy()
        )

        distance_matrix = pairwise_distances(
            X_bin,
            metric="euclidean",
        )

        mileage_bin = (
            bin_df["mileage"]
            .to_numpy(dtype=float)
        )

        mileage_diff_matrix = np.abs(
            mileage_bin[:, None]
            - mileage_bin[None, :]
        )

        allowed_mask = (
            mileage_diff_matrix
            <= MAX_MILEAGE_DIFF
        )

        np.fill_diagonal(
            allowed_mask,
            False,
        )

        constrained_distances = (
            distance_matrix.copy()
        )

        constrained_distances[
            ~allowed_mask
        ] = np.inf

        for position, global_idx in enumerate(
            bin_indices
        ):

            valid_positions = np.where(
                np.isfinite(
                    constrained_distances[
                        position
                    ]
                )
            )[0]

            if len(valid_positions) == 0:

                selected_neighbors = []

            else:

                ordered_positions = (
                    valid_positions[
                        np.argsort(
                            constrained_distances[
                                position,
                                valid_positions,
                            ]
                        )
                    ]
                )

                selected_positions = (
                    ordered_positions[
                        :K_NEIGHBORS_SMOTER
                    ]
                )

                selected_neighbors = (
                    bin_indices[
                        selected_positions
                    ]
                    .astype(int)
                    .tolist()
                )

            neighbor_map[
                int(global_idx)
            ] = selected_neighbors

            neighbor_audit_rows.append(
                {
                    "fold": int(fold_id),
                    "local_index": int(
                        global_idx
                    ),
                    "event_id": eligible.loc[
                        global_idx,
                        "event_id",
                    ],
                    "soh_bin": str(
                        soh_bin_value
                    ),
                    "mileage": float(
                        eligible.loc[
                            global_idx,
                            "mileage",
                        ]
                    ),
                    "n_candidates_mileage": int(
                        len(valid_positions)
                    ),
                    "n_neighbors_available": int(
                        len(selected_neighbors)
                    ),
                }
            )

    neighbor_audit_fold = pd.DataFrame(
        neighbor_audit_rows
    )

    # ------------------------------------------------------
    # 4.5. Plano de balanceamento
    # ------------------------------------------------------

    train_bin_counts = (
        train_real_fold
        .groupby(
            "soh_bin",
            observed=True,
        )
        .size()
        .rename("n_real_train")
    )

    balance_plan_rows = []

    valid_seeds_by_bin = {}

    for soh_bin_value, n_real in (
        train_bin_counts.items()
    ):

        target_final = definir_alvo_smoter(
            soh_bin_value
        )

        eligible_indices = (
            eligible.index[
                eligible["soh_bin"]
                == soh_bin_value
            ]
            .tolist()
        )

        valid_seed_indices = [
            int(idx)
            for idx in eligible_indices
            if len(
                neighbor_map.get(
                    int(idx),
                    [],
                )
            ) > 0
        ]

        valid_seeds_by_bin[
            soh_bin_value
        ] = valid_seed_indices

        if target_final is None:

            action = "manter"
            n_requested = 0

        elif int(n_real) >= int(target_final):

            action = "manter_acima_do_alvo"
            n_requested = 0

        else:

            action = "gerar_smoter"
            n_requested = int(
                target_final
                - n_real
            )

        balance_plan_rows.append(
            {
                "fold": int(fold_id),
                "soh_bin": str(
                    soh_bin_value
                ),
                "n_real_train": int(n_real),
                "target_final": (
                    int(target_final)
                    if target_final is not None
                    else np.nan
                ),
                "n_synthetic_requested": int(
                    n_requested
                ),
                "n_eligible_mileage": int(
                    len(eligible_indices)
                ),
                "n_valid_seeds": int(
                    len(valid_seed_indices)
                ),
                "action": action,
            }
        )

    balance_plan_fold = pd.DataFrame(
        balance_plan_rows
    )

    invalid_bins = (
        balance_plan_fold.loc[
            (
                balance_plan_fold[
                    "n_synthetic_requested"
                ] > 0
            )
            &
            (
                balance_plan_fold[
                    "n_valid_seeds"
                ] == 0
            )
        ]
    )

    if not invalid_bins.empty:

        display(invalid_bins)

        raise ValueError(
            f"Fold {fold_id}: há bins que exigem "
            "sintéticos, mas não possuem sementes "
            "com vizinhos válidos."
        )

    # ------------------------------------------------------
    # 4.6. Geração dos eventos sintéticos
    # ------------------------------------------------------

    synthetic_rows = []
    generation_audit_rows = []

    synthetic_counter = 0

    for _, plan_row in (
        balance_plan_fold.iterrows()
    ):

        n_to_generate = int(
            plan_row[
                "n_synthetic_requested"
            ]
        )

        if n_to_generate <= 0:
            continue

        # Recuperar o intervalo original a partir do texto
        matching_bins = [
            interval
            for interval in train_bin_counts.index
            if str(interval)
            == plan_row["soh_bin"]
        ]

        if len(matching_bins) != 1:
            raise ValueError(
                f"Fold {fold_id}: não foi possível "
                f"recuperar o bin {plan_row['soh_bin']}."
            )

        soh_bin_value = matching_bins[0]

        valid_seeds = (
            valid_seeds_by_bin[
                soh_bin_value
            ]
        )

        for _ in range(n_to_generate):

            seed_idx = int(
                rng.choice(
                    valid_seeds
                )
            )

            neighbor_idx = int(
                rng.choice(
                    neighbor_map[
                        seed_idx
                    ]
                )
            )

            interpolation_factor = float(
                rng.random()
            )

            parent_a = eligible.loc[
                seed_idx
            ]

            parent_b = eligible.loc[
                neighbor_idx
            ]

            synthetic_features = (
                X_original.loc[
                    seed_idx
                ]
                +
                interpolation_factor
                * (
                    X_original.loc[
                        neighbor_idx
                    ]
                    -
                    X_original.loc[
                        seed_idx
                    ]
                )
            )

            synthetic_soh = float(
                parent_a[TARGET_COL]
                +
                interpolation_factor
                * (
                    parent_b[TARGET_COL]
                    -
                    parent_a[TARGET_COL]
                )
            )

            synthetic_mileage = float(
                parent_a["mileage"]
                +
                interpolation_factor
                * (
                    parent_b["mileage"]
                    -
                    parent_a["mileage"]
                )
            )

            synthetic_counter += 1

            synthetic_event_id = (
                f"smoter_f{fold_id}_"
                f"{synthetic_counter:06d}"
            )

            synthetic_row = {
                column: np.nan
                for column
                in train_real_fold.columns
            }

            # Copiar metadados do pai A como referência.
            # Campos interpolados serão sobrescritos abaixo.
            for column in train_real_fold.columns:

                if column in parent_a.index:
                    synthetic_row[column] = (
                        parent_a[column]
                    )

            for feature in feature_cols:

                synthetic_row[feature] = float(
                    synthetic_features[
                        feature
                    ]
                )

            synthetic_row["event_id"] = (
                synthetic_event_id
            )

            synthetic_row[TARGET_COL] = (
                synthetic_soh
            )

            synthetic_row["mileage"] = (
                synthetic_mileage
            )

            synthetic_row["soh_bin"] = (
                soh_bin_value
            )

            synthetic_row["sample_origin"] = (
                "smoter"
            )

            synthetic_row["split"] = (
                "train_smoter"
            )

            synthetic_row["source_fold"] = int(
                fold_id
            )

            # Não atribuir o sintético a um carro real.
            synthetic_row["car"] = (
                f"SMOTER_F{fold_id}"
            )

            synthetic_row["charge_segment"] = (
                "synthetic"
            )

            synthetic_rows.append(
                synthetic_row
            )

            generation_audit_rows.append(
                {
                    "fold": int(fold_id),
                    "synthetic_event_id":
                        synthetic_event_id,

                    "soh_bin":
                        str(soh_bin_value),

                    "parent_a_event_id":
                        parent_a["event_id"],

                    "parent_b_event_id":
                        parent_b["event_id"],

                    "parent_a_soh":
                        float(
                            parent_a[TARGET_COL]
                        ),

                    "parent_b_soh":
                        float(
                            parent_b[TARGET_COL]
                        ),

                    "synthetic_soh":
                        synthetic_soh,

                    "parent_a_mileage":
                        float(
                            parent_a["mileage"]
                        ),

                    "parent_b_mileage":
                        float(
                            parent_b["mileage"]
                        ),

                    "synthetic_mileage":
                        synthetic_mileage,

                    "parent_mileage_diff":
                        float(
                            abs(
                                parent_a["mileage"]
                                - parent_b["mileage"]
                            )
                        ),

                    "interpolation_factor":
                        interpolation_factor,
                }
            )

    synthetic_fold = pd.DataFrame(
        synthetic_rows
    )

    generation_audit_fold = pd.DataFrame(
        generation_audit_rows
    )

    # ------------------------------------------------------
    # 4.7. Construção do treino balanceado
    # ------------------------------------------------------

    if len(synthetic_fold) > 0:

        synthetic_fold = synthetic_fold[
            train_real_fold.columns
        ].copy()

        train_balanced_fold = pd.concat(
            [
                train_real_fold,
                synthetic_fold,
            ],
            axis=0,
            ignore_index=True,
        )

    else:

        synthetic_fold = pd.DataFrame(
            columns=train_real_fold.columns
        )

        train_balanced_fold = (
            train_real_fold.copy()
        )

    # ------------------------------------------------------
    # 4.8. Auditorias finais
    # ------------------------------------------------------

    if not train_balanced_fold[
        "event_id"
    ].is_unique:

        raise ValueError(
            f"Fold {fold_id}: existem event_id duplicados "
            "no treino balanceado."
        )

    expected_synthetic = int(
        balance_plan_fold[
            "n_synthetic_requested"
        ].sum()
    )

    if len(synthetic_fold) != expected_synthetic:

        raise ValueError(
            f"Fold {fold_id}: foram gerados "
            f"{len(synthetic_fold)} sintéticos, "
            f"mas eram esperados "
            f"{expected_synthetic}."
        )

    if len(generation_audit_fold) > 0:

        max_parent_difference = float(
            generation_audit_fold[
                "parent_mileage_diff"
            ].max()
        )

        if (
            max_parent_difference
            > MAX_MILEAGE_DIFF + 1e-9
        ):
            raise ValueError(
                f"Fold {fold_id}: foi encontrado par "
                "de pais com diferença de mileage "
                "acima do permitido."
            )

    return {
        "train_balanced":
            train_balanced_fold,

        "synthetic":
            synthetic_fold,

        "balance_plan":
            balance_plan_fold,

        "neighbor_audit":
            neighbor_audit_fold,

        "generation_audit":
            generation_audit_fold,

        "constant_features":
            constant_features,

        "distance_features":
            smoter_distance_features,

        "feature_medians":
            feature_medians,

        "scaler":
            scaler,
    }


print("\nFunção aplicar_smoter_fold criada com sucesso.")
print("Mileage mínima dos pais:", MILEAGE_SMOTER_MIN)
print("Diferença máxima entre pais:", MAX_MILEAGE_DIFF)
print("Máximo de vizinhos:", K_NEIGHBORS_SMOTER)
print("Alvo primeiro bin:", TARGET_FIRST_BIN)
print("Alvo demais bins:", TARGET_OTHER_BINS)

CONFIGURAÇÃO DO SMOTER
Features físicas oficiais: 250

Função aplicar_smoter_fold criada com sucesso.
Mileage mínima dos pais: 30000
Diferença máxima entre pais: 15000
Máximo de vizinhos: 5
Alvo primeiro bin: 300
Alvo demais bins: 500


## 8. Aplicação do SMOTER nos cinco folds

Para cada fold:

1. o conjunto de validação é separado e preservado com eventos exclusivamente reais;
2. o SMOTER é ajustado e aplicado apenas ao conjunto de treino;
3. as amostras sintéticas são adicionadas ao treino real;
4. são realizadas auditorias de origem, distribuição por SOH, mileage e vazamento de `event_id`;
5. os arquivos de cada fold são salvos para permitir a reprodução do treinamento.

O balanceamento não utiliza nenhuma informação dos eventos pertencentes ao fold de validação.

In [13]:
# ==========================================================
# 8. APLICAR O SMOTER NOS CINCO FOLDS
# ==========================================================

import gc
import time

# ----------------------------------------------------------
# 1. Estruturas para armazenar os resultados
# ----------------------------------------------------------

CV_FOLD_DATA = {}

smoter_fold_summary_rows = []
smoter_balance_plan_all = []
smoter_neighbor_audit_all = []
smoter_generation_audit_all = []

smoter_total_start = time.perf_counter()

# ----------------------------------------------------------
# 2. Processar cada fold
# ----------------------------------------------------------

for fold_id in range(N_SPLITS):

    fold_start = time.perf_counter()

    print("\n" + "=" * 100)
    print(f"PROCESSANDO FOLD {fold_id}")
    print("=" * 100)

    # ------------------------------------------------------
    # 2.1. Separar treino real e validação real
    # ------------------------------------------------------

    valid_mask = df_cv["cv_fold"].eq(fold_id)

    train_real_fold = (
        df_cv.loc[~valid_mask]
        .copy()
        .reset_index(drop=True)
    )

    valid_real_fold = (
        df_cv.loc[valid_mask]
        .copy()
        .reset_index(drop=True)
    )

    train_real_fold["split"] = "train_real"
    train_real_fold["source_fold"] = fold_id
    train_real_fold["sample_origin"] = "real"

    valid_real_fold["split"] = "validation_real"
    valid_real_fold["source_fold"] = fold_id
    valid_real_fold["sample_origin"] = "real"

    print("Treino real    :", len(train_real_fold))
    print("Validação real :", len(valid_real_fold))

    # ------------------------------------------------------
    # 2.2. Auditoria antes do SMOTER
    # ------------------------------------------------------

    shared_event_ids_before = set(
        train_real_fold["event_id"]
    ).intersection(
        set(valid_real_fold["event_id"])
    )

    if shared_event_ids_before:
        raise ValueError(
            f"Fold {fold_id}: existem event_id compartilhados "
            "entre treino e validação antes do SMOTER."
        )

    if not valid_real_fold["sample_origin"].eq("real").all():
        raise ValueError(
            f"Fold {fold_id}: a validação contém eventos "
            "não reais."
        )

    # ------------------------------------------------------
    # 2.3. Aplicar o SMOTER somente ao treino
    # ------------------------------------------------------

    smoter_result = aplicar_smoter_fold(
        train_real_fold=train_real_fold,
        fold_id=fold_id,
        feature_cols=FEATURE_COLS,
        random_state=SMOTER_RANDOM_STATE,
    )

    train_balanced_fold = (
        smoter_result["train_balanced"]
        .copy()
        .reset_index(drop=True)
    )

    synthetic_fold = (
        smoter_result["synthetic"]
        .copy()
        .reset_index(drop=True)
    )

    balance_plan_fold = (
        smoter_result["balance_plan"]
        .copy()
    )

    neighbor_audit_fold = (
        smoter_result["neighbor_audit"]
        .copy()
    )

    generation_audit_fold = (
        smoter_result["generation_audit"]
        .copy()
    )

    # ------------------------------------------------------
    # 2.4. Auditorias após o SMOTER
    # ------------------------------------------------------

    n_train_real = len(train_real_fold)
    n_synthetic = len(synthetic_fold)
    n_train_balanced = len(train_balanced_fold)
    n_valid_real = len(valid_real_fold)

    if n_train_balanced != n_train_real + n_synthetic:
        raise ValueError(
            f"Fold {fold_id}: tamanho do treino balanceado "
            "inconsistente."
        )

    origin_counts_train = (
        train_balanced_fold["sample_origin"]
        .value_counts()
        .to_dict()
    )

    n_real_in_balanced = int(
        origin_counts_train.get("real", 0)
    )

    n_smoter_in_balanced = int(
        origin_counts_train.get("smoter", 0)
    )

    if n_real_in_balanced != n_train_real:
        raise ValueError(
            f"Fold {fold_id}: número de eventos reais no "
            "treino balanceado está incorreto."
        )

    if n_smoter_in_balanced != n_synthetic:
        raise ValueError(
            f"Fold {fold_id}: número de eventos sintéticos no "
            "treino balanceado está incorreto."
        )

    if not valid_real_fold["sample_origin"].eq("real").all():
        raise ValueError(
            f"Fold {fold_id}: a validação foi contaminada "
            "por eventos sintéticos."
        )

    shared_event_ids_after = set(
        train_balanced_fold["event_id"]
    ).intersection(
        set(valid_real_fold["event_id"])
    )

    if shared_event_ids_after:
        raise ValueError(
            f"Fold {fold_id}: vazamento de event_id entre "
            "treino balanceado e validação."
        )

    # Nenhum ID sintético pode coincidir com evento real
    synthetic_real_collision = set(
        synthetic_fold["event_id"]
    ).intersection(
        set(df_cv["event_id"])
    )

    if synthetic_real_collision:
        raise ValueError(
            f"Fold {fold_id}: algum event_id sintético "
            "coincide com um evento real."
        )

    # Conferência da faixa de SOH
    if not train_balanced_fold[TARGET_COL].between(
        SOH_MIN,
        SOH_MAX,
        inclusive="both",
    ).all():
        raise ValueError(
            f"Fold {fold_id}: o treino balanceado contém "
            "SOH fora da faixa oficial."
        )

    if not valid_real_fold[TARGET_COL].between(
        SOH_MIN,
        SOH_MAX,
        inclusive="both",
    ).all():
        raise ValueError(
            f"Fold {fold_id}: a validação contém SOH fora "
            "da faixa oficial."
        )

    # Conferência das features usadas pelos modelos
    required_model_features = list(
        dict.fromkeys(
            FEATURES_GLOBAL
            + FEATURES_EXPERTS
            + FEATURES_GATE
        )
    )

    for dataset_name, dataset_fold in {
        "train_balanced": train_balanced_fold,
        "validation_real": valid_real_fold,
    }.items():

        invalid_count = int(
            dataset_fold[required_model_features]
            .apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .isna()
            .sum()
            .sum()
        )

        if invalid_count > 0:
            raise ValueError(
                f"Fold {fold_id} — {dataset_name}: existem "
                f"{invalid_count} NaN/Inf nas features finais."
            )

    # ------------------------------------------------------
    # 2.5. Distribuições antes e depois
    # ------------------------------------------------------

    distribution_train_real = (
        train_real_fold
        .groupby("soh_bin", observed=False)
        .size()
        .rename("n_train_real")
    )

    distribution_train_balanced = (
        train_balanced_fold
        .groupby("soh_bin", observed=False)
        .size()
        .rename("n_train_balanced")
    )

    distribution_valid_real = (
        valid_real_fold
        .groupby("soh_bin", observed=False)
        .size()
        .rename("n_valid_real")
    )

    fold_distribution = pd.concat(
        [
            distribution_train_real,
            distribution_train_balanced,
            distribution_valid_real,
        ],
        axis=1,
    ).fillna(0)

    fold_distribution = (
        fold_distribution
        .astype(int)
        .reset_index()
    )

    fold_distribution.insert(
        0,
        "fold",
        fold_id,
    )

    fold_distribution["n_synthetic"] = (
        fold_distribution["n_train_balanced"]
        - fold_distribution["n_train_real"]
    )

    # ------------------------------------------------------
    # 2.6. Diretório e salvamento do fold
    # ------------------------------------------------------

    fold_dir = FOLDS_DIR / f"fold_{fold_id}"

    fold_data_dir = fold_dir / "data"
    fold_audit_dir = fold_dir / "audit"

    fold_data_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    fold_audit_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    train_real_fold.to_csv(
        fold_data_dir / "train_real.csv",
        index=False,
    )

    synthetic_fold.to_csv(
        fold_data_dir / "train_smoter_synthetic.csv",
        index=False,
    )

    train_balanced_fold.to_csv(
        fold_data_dir / "train_balanced.csv",
        index=False,
    )

    valid_real_fold.to_csv(
        fold_data_dir / "validation_real.csv",
        index=False,
    )

    balance_plan_fold.to_excel(
        fold_audit_dir / "smoter_balance_plan.xlsx",
        index=False,
    )

    neighbor_audit_fold.to_csv(
        fold_audit_dir / "smoter_neighbor_audit.csv",
        index=False,
    )

    generation_audit_fold.to_csv(
        fold_audit_dir / "smoter_generation_audit.csv",
        index=False,
    )

    fold_distribution.to_excel(
        fold_audit_dir / "distribution_before_after_smoter.xlsx",
        index=False,
    )

    # ------------------------------------------------------
    # 2.7. Guardar em memória
    # ------------------------------------------------------

    CV_FOLD_DATA[fold_id] = {
        "train_real": train_real_fold,
        "train_balanced": train_balanced_fold,
        "synthetic": synthetic_fold,
        "validation_real": valid_real_fold,
        "balance_plan": balance_plan_fold,
        "fold_distribution": fold_distribution,
        "constant_features": (
            smoter_result["constant_features"]
        ),
        "distance_features": (
            smoter_result["distance_features"]
        ),
    }

    # ------------------------------------------------------
    # 2.8. Consolidar auditorias
    # ------------------------------------------------------

    smoter_balance_plan_all.append(
        balance_plan_fold
    )

    smoter_neighbor_audit_all.append(
        neighbor_audit_fold
    )

    if len(generation_audit_fold) > 0:
        smoter_generation_audit_all.append(
            generation_audit_fold
        )

    max_parent_mileage_diff = (
        float(
            generation_audit_fold[
                "parent_mileage_diff"
            ].max()
        )
        if len(generation_audit_fold) > 0
        else np.nan
    )

    fold_time = (
        time.perf_counter()
        - fold_start
    )

    smoter_fold_summary_rows.append(
        {
            "fold": fold_id,
            "n_train_real": n_train_real,
            "n_synthetic": n_synthetic,
            "n_train_balanced": n_train_balanced,
            "n_valid_real": n_valid_real,
            "n_features_distance": len(
                smoter_result["distance_features"]
            ),
            "n_constant_features": len(
                smoter_result["constant_features"]
            ),
            "event_ids_shared": len(
                shared_event_ids_after
            ),
            "validation_synthetic_events": int(
                valid_real_fold["sample_origin"]
                .ne("real")
                .sum()
            ),
            "max_parent_mileage_diff": (
                max_parent_mileage_diff
            ),
            "soh_synthetic_min": (
                synthetic_fold[TARGET_COL].min()
                if n_synthetic > 0
                else np.nan
            ),
            "soh_synthetic_max": (
                synthetic_fold[TARGET_COL].max()
                if n_synthetic > 0
                else np.nan
            ),
            "mileage_synthetic_min": (
                synthetic_fold["mileage"].min()
                if n_synthetic > 0
                else np.nan
            ),
            "mileage_synthetic_max": (
                synthetic_fold["mileage"].max()
                if n_synthetic > 0
                else np.nan
            ),
            "tempo_smoter_s": fold_time,
        }
    )

    print("\nResumo do fold:")
    print("Treino real       :", n_train_real)
    print("Sintéticos gerados:", n_synthetic)
    print("Treino balanceado :", n_train_balanced)
    print("Validação real    :", n_valid_real)
    print(
        "Maior diferença mileage entre pais:",
        max_parent_mileage_diff,
    )
    print(
        "Tempo:",
        round(fold_time, 2),
        "s",
    )

    gc.collect()

# ----------------------------------------------------------
# 3. Consolidar auditorias dos cinco folds
# ----------------------------------------------------------

smoter_fold_summary = pd.DataFrame(
    smoter_fold_summary_rows
)

smoter_balance_plan_cv = pd.concat(
    smoter_balance_plan_all,
    axis=0,
    ignore_index=True,
)

smoter_neighbor_audit_cv = pd.concat(
    smoter_neighbor_audit_all,
    axis=0,
    ignore_index=True,
)

if smoter_generation_audit_all:

    smoter_generation_audit_cv = pd.concat(
        smoter_generation_audit_all,
        axis=0,
        ignore_index=True,
    )

else:

    smoter_generation_audit_cv = pd.DataFrame()

smoter_total_time = (
    time.perf_counter()
    - smoter_total_start
)

# ----------------------------------------------------------
# 4. Auditorias consolidadas obrigatórias
# ----------------------------------------------------------

if smoter_fold_summary["event_ids_shared"].sum() != 0:
    raise ValueError(
        "Foi detectado vazamento de eventos em algum fold."
    )

if (
    smoter_fold_summary[
        "validation_synthetic_events"
    ].sum()
    != 0
):
    raise ValueError(
        "Foi detectado evento sintético em uma validação."
    )

if (
    smoter_fold_summary[
        "max_parent_mileage_diff"
    ]
    .dropna()
    .gt(MAX_MILEAGE_DIFF + 1e-9)
    .any()
):
    raise ValueError(
        "Algum par de pais ultrapassou a diferença "
        "máxima de mileage."
    )

# Cada evento real deve aparecer uma vez nas validações
validation_event_ids_all = pd.concat(
    [
        CV_FOLD_DATA[fold_id][
            "validation_real"
        ][["event_id"]]
        for fold_id in range(N_SPLITS)
    ],
    axis=0,
    ignore_index=True,
)

if len(validation_event_ids_all) != len(df_cv):
    raise ValueError(
        "A soma das validações não possui o mesmo número "
        "de eventos do dataset."
    )

if not validation_event_ids_all["event_id"].is_unique:
    raise ValueError(
        "Algum evento aparece em mais de uma validação."
    )

if set(validation_event_ids_all["event_id"]) != set(
    df_cv["event_id"]
):
    raise ValueError(
        "Os eventos das validações não correspondem "
        "exatamente ao Dataset Gold limpo."
    )

# ----------------------------------------------------------
# 5. Salvar auditorias consolidadas
# ----------------------------------------------------------

smoter_fold_summary.to_excel(
    AUDIT_DIR / "smoter_cv_fold_summary.xlsx",
    index=False,
)

smoter_balance_plan_cv.to_excel(
    AUDIT_DIR / "smoter_cv_balance_plan.xlsx",
    index=False,
)

smoter_neighbor_audit_cv.to_csv(
    AUDIT_DIR / "smoter_cv_neighbor_audit.csv",
    index=False,
)

smoter_generation_audit_cv.to_csv(
    AUDIT_DIR / "smoter_cv_generation_audit.csv",
    index=False,
)

# ----------------------------------------------------------
# 6. Exibir resultados
# ----------------------------------------------------------

print("\n" + "=" * 100)
print("SMOTER CONCLUÍDO NOS CINCO FOLDS")
print("=" * 100)

display(smoter_fold_summary)

print(
    "\nTotal de eventos sintéticos gerados:",
    int(smoter_fold_summary["n_synthetic"].sum()),
)

print(
    "Tempo total:",
    round(smoter_total_time / 60, 2),
    "min",
)

print("\nDados dos folds disponíveis em:")
print("CV_FOLD_DATA[fold_id]")

print("\nArquivos salvos em:")
print(FOLDS_DIR)


PROCESSANDO FOLD 0
Treino real    : 9207
Validação real : 2302

Resumo do fold:
Treino real       : 9207
Sintéticos gerados: 2835
Treino balanceado : 12042
Validação real    : 2302
Maior diferença mileage entre pais: 14967.215999999993
Tempo: 28.99 s

PROCESSANDO FOLD 1
Treino real    : 9207
Validação real : 2302

Resumo do fold:
Treino real       : 9207
Sintéticos gerados: 2834
Treino balanceado : 12041
Validação real    : 2302
Maior diferença mileage entre pais: 14983.689599999998
Tempo: 29.41 s

PROCESSANDO FOLD 2
Treino real    : 9207
Validação real : 2302

Resumo do fold:
Treino real       : 9207
Sintéticos gerados: 2836
Treino balanceado : 12043
Validação real    : 2302
Maior diferença mileage entre pais: 14990.131200000003
Tempo: 30.96 s

PROCESSANDO FOLD 3
Treino real    : 9207
Validação real : 2302

Resumo do fold:
Treino real       : 9207
Sintéticos gerados: 2836
Treino balanceado : 12043
Validação real    : 2302
Maior diferença mileage entre pais: 14993.721599999997
Tempo: 

,fold,n_train_real,n_synthetic,n_train_balanced,n_valid_real,n_features_distance,n_constant_features,event_ids_shared,validation_synthetic_events,max_parent_mileage_diff,soh_synthetic_min,soh_synthetic_max,mileage_synthetic_min,mileage_synthetic_max,tempo_smoter_s
0,0,9207,2835,12042,2302,250,0,0,0,14967.2160,0.920020,1.004651,30236.387985,98744.704692,28.985762
1,1,9207,2834,12041,2302,250,0,0,0,14983.6896,0.920033,1.004822,30065.258576,100983.465485,29.406702
2,2,9207,2836,12043,2302,250,0,0,0,14990.1312,0.920449,1.004954,30089.766660,101333.751040,30.958695
3,3,9207,2836,12043,2302,250,0,0,0,14993.7216,0.920178,1.004822,30549.018403,100036.942934,28.846340
4,4,9208,2835,12043,2301,250,0,0,0,14975.2416,0.920015,1.004784,30087.077665,99905.015638,28.645429



Total de eventos sintéticos gerados: 14176
Tempo total: 2.45 min

Dados dos folds disponíveis em:
CV_FOLD_DATA[fold_id]

Arquivos salvos em:
C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\OUTPUT\folds


## 9. Configuração dos modelos finais

Nesta etapa são definidas as configurações fixas dos modelos avaliados.

### Modelo global

- `ExtraTreesRegressor`;
- 20 features físicas campeãs;
- `mileage`;
- 21 entradas.

### Especialistas

| Especialista | Faixa de SOH | Modelo |
|---|---:|---|
| A — baixo | 0,920 a 0,945 | XGBoost |
| B — intermediário | 0,940 a 0,975 | ExtraTrees |
| C — alto | 0,970 a 1,005 | ExtraTrees |

Todos os especialistas utilizam as mesmas 20 features físicas do modelo global, acrescidas de `mileage`.

### Gate do Mixture of Experts

- `XGBClassifier`;
- 75 features limpas;
- `mileage`;
- 76 entradas;
- três classes: A, B e C.

Os rótulos de treinamento do gate serão construídos com predições out-of-fold dos especialistas, evitando que o gate seja treinado com erros excessivamente otimistas.

In [14]:
# ==========================================================
# 9. CONFIGURAÇÃO DOS MODELOS E FUNÇÕES AUXILIARES
# ==========================================================

import copy
import json
import time

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)

from xgboost import (
    XGBClassifier,
    XGBRegressor,
)

# ----------------------------------------------------------
# 1. Parâmetros fixos do experimento
# ----------------------------------------------------------

MODEL_RANDOM_STATE = RANDOM_STATE

INNER_OOF_SPLITS = 5

ROUTER_BOUNDARY_A_B = 0.9425
ROUTER_BOUNDARY_B_C = 0.9725

EXPERT_CLASS_NAMES = {
    0: "A_XGBoost_Baixo",
    1: "B_ExtraTrees_Intermediario",
    2: "C_ExtraTrees_Alto",
}

# ----------------------------------------------------------
# 2. Configurações dos modelos
# ----------------------------------------------------------

GLOBAL_MODEL_PARAMS = {
    "n_estimators": 1200,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "bootstrap": False,
    "random_state": MODEL_RANDOM_STATE,
    "n_jobs": -1,
}

EXPERT_A_XGB_PARAMS = {
    "n_estimators": 1200,
    "learning_rate": 0.02,
    "max_depth": 6,
    "min_child_weight": 2,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.05,
    "reg_lambda": 1.5,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "random_state": MODEL_RANDOM_STATE,
    "n_jobs": -1,
}

EXPERT_BC_ET_PARAMS = {
    "n_estimators": 1000,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "bootstrap": False,
    "random_state": MODEL_RANDOM_STATE,
    "n_jobs": -1,
}

GATE_XGB_PARAMS = {
    "n_estimators": 1200,
    "learning_rate": 0.02,
    "max_depth": 6,
    "min_child_weight": 2,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.05,
    "reg_lambda": 1.5,
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "random_state": MODEL_RANDOM_STATE,
    "n_jobs": -1,
}

# ----------------------------------------------------------
# 3. Funções construtoras
# ----------------------------------------------------------

def criar_modelo_global(
    random_state=MODEL_RANDOM_STATE,
):
    """Cria o ExtraTrees global campeão."""

    params = copy.deepcopy(
        GLOBAL_MODEL_PARAMS
    )

    params["random_state"] = int(
        random_state
    )

    return ExtraTreesRegressor(
        **params
    )


def criar_especialista(
    class_id,
    random_state=MODEL_RANDOM_STATE,
):
    """
    Cria um dos três especialistas.

    0 = XGBoost baixo
    1 = ExtraTrees intermediário
    2 = ExtraTrees alto
    """

    if class_id == 0:

        params = copy.deepcopy(
            EXPERT_A_XGB_PARAMS
        )

        params["random_state"] = int(
            random_state
        )

        return XGBRegressor(
            **params
        )

    if class_id in {1, 2}:

        params = copy.deepcopy(
            EXPERT_BC_ET_PARAMS
        )

        params["random_state"] = int(
            random_state
        )

        return ExtraTreesRegressor(
            **params
        )

    raise ValueError(
        f"class_id inválido: {class_id}"
    )


def criar_gate_xgboost(
    random_state=MODEL_RANDOM_STATE,
):
    """Cria o gate XGBoost de três classes."""

    params = copy.deepcopy(
        GATE_XGB_PARAMS
    )

    params["random_state"] = int(
        random_state
    )

    return XGBClassifier(
        **params
    )

# ----------------------------------------------------------
# 4. Função para obter a máscara da faixa de um especialista
# ----------------------------------------------------------

def obter_mascara_especialista(
    dataframe,
    class_id,
    target_col=TARGET_COL,
):
    """Seleciona os eventos da faixa real do especialista."""

    if class_id not in EXPERT_CONFIG:
        raise ValueError(
            f"Especialista inexistente: {class_id}"
        )

    soh_min = float(
        EXPERT_CONFIG[class_id]["soh_min"]
    )

    soh_max = float(
        EXPERT_CONFIG[class_id]["soh_max"]
    )

    return dataframe[target_col].between(
        soh_min,
        soh_max,
        inclusive="both",
    )

# ----------------------------------------------------------
# 5. Função de avaliação de regressão
# ----------------------------------------------------------

def calcular_metricas_regressao(
    y_true,
    y_pred,
):
    """Calcula as métricas oficiais de regressão."""

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    if y_true.shape != y_pred.shape:
        raise ValueError(
            "y_true e y_pred possuem shapes diferentes: "
            f"{y_true.shape} e {y_pred.shape}."
        )

    if not np.isfinite(y_true).all():
        raise ValueError(
            "y_true contém NaN ou Inf."
        )

    if not np.isfinite(y_pred).all():
        raise ValueError(
            "y_pred contém NaN ou Inf."
        )

    error = y_pred - y_true
    absolute_error = np.abs(error)

    return {
        "rmse": float(
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred,
                )
            )
        ),

        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "r2": float(
            r2_score(
                y_true,
                y_pred,
            )
        ),

        "mape_%": float(
            np.mean(
                absolute_error
                / np.clip(
                    np.abs(y_true),
                    1e-12,
                    None,
                )
            )
            * 100
        ),

        "bias": float(
            np.mean(error)
        ),

        "erro_abs_p95": float(
            np.percentile(
                absolute_error,
                95,
            )
        ),

        "erro_abs_max": float(
            np.max(
                absolute_error
            )
        ),
    }

# ----------------------------------------------------------
# 6. Função de avaliação do classificador do gate
# ----------------------------------------------------------

def calcular_metricas_gate(
    y_true,
    y_pred,
    y_proba=None,
):
    """Calcula as métricas oficiais do gate."""

    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    result = {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "f1_weighted": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
    }

    if y_proba is not None:

        y_proba = np.asarray(
            y_proba,
            dtype=float,
        )

        result["log_loss"] = float(
            log_loss(
                y_true,
                y_proba,
                labels=[0, 1, 2],
            )
        )

    else:

        result["log_loss"] = np.nan

    return result

# ----------------------------------------------------------
# 7. Seleção das predições pelo gate
# ----------------------------------------------------------

def combinar_predicoes_especialistas(
    expert_prediction_matrix,
    selected_classes,
):
    """
    Seleciona uma das três predições de especialista
    para cada evento.
    """

    matrix = np.asarray(
        expert_prediction_matrix,
        dtype=float,
    )

    classes = np.asarray(
        selected_classes,
        dtype=int,
    )

    if matrix.ndim != 2:
        raise ValueError(
            "A matriz dos especialistas deve ser 2D."
        )

    if matrix.shape[1] != 3:
        raise ValueError(
            "A matriz deve possuir três colunas."
        )

    if len(classes) != matrix.shape[0]:
        raise ValueError(
            "O vetor de classes não está alinhado "
            "com a matriz dos especialistas."
        )

    if not np.isin(
        classes,
        [0, 1, 2],
    ).all():
        raise ValueError(
            "Existem classes inválidas no roteamento."
        )

    row_indices = np.arange(
        matrix.shape[0]
    )

    return matrix[
        row_indices,
        classes,
    ]

# ----------------------------------------------------------
# 8. Roteamento pela previsão do modelo global
# ----------------------------------------------------------

def rotear_pela_previsao_global(
    global_predictions,
    boundary_ab=ROUTER_BOUNDARY_A_B,
    boundary_bc=ROUTER_BOUNDARY_B_C,
):
    """
    Converte a previsão contínua do modelo global
    em classe de especialista.
    """

    predictions = np.asarray(
        global_predictions,
        dtype=float,
    )

    return np.select(
        [
            predictions < boundary_ab,
            predictions < boundary_bc,
        ],
        [
            0,
            1,
        ],
        default=2,
    ).astype(int)

# ----------------------------------------------------------
# 9. Oracle dos especialistas
# ----------------------------------------------------------

def calcular_oracle_especialistas(
    y_true,
    expert_prediction_matrix,
):
    """
    Escolhe o especialista de menor erro absoluto para
    cada evento.

    Utilizado apenas para avaliação e para construção
    dos rótulos do gate.
    """

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    matrix = np.asarray(
        expert_prediction_matrix,
        dtype=float,
    )

    if matrix.shape != (
        len(y_true),
        3,
    ):
        raise ValueError(
            "Shape inválido da matriz dos especialistas: "
            f"{matrix.shape}."
        )

    absolute_errors = np.abs(
        matrix
        - y_true[:, None]
    )

    oracle_classes = np.argmin(
        absolute_errors,
        axis=1,
    ).astype(int)

    oracle_predictions = (
        combinar_predicoes_especialistas(
            matrix,
            oracle_classes,
        )
    )

    sorted_errors = np.sort(
        absolute_errors,
        axis=1,
    )

    expert_margin = (
        sorted_errors[:, 1]
        - sorted_errors[:, 0]
    )

    denominator = np.maximum(
        sorted_errors[:, 1],
        1e-12,
    )

    oracle_confidence = np.clip(
        expert_margin / denominator,
        0.0,
        1.0,
    )

    return {
        "classes": oracle_classes,
        "predictions": oracle_predictions,
        "absolute_errors": absolute_errors,
        "expert_margin": expert_margin,
        "oracle_confidence": oracle_confidence,
    }

# ----------------------------------------------------------
# 10. Auditoria rápida das funções e dos modelos
# ----------------------------------------------------------

model_audit = pd.DataFrame(
    [
        {
            "componente": "Global",
            "modelo": criar_modelo_global(
            ).__class__.__name__,
            "n_features": len(
                FEATURES_GLOBAL
            ),
            "faixa_soh": "0.920–1.005",
        },
        {
            "componente": "Especialista A",
            "modelo": criar_especialista(
                0
            ).__class__.__name__,
            "n_features": len(
                FEATURES_EXPERTS
            ),
            "faixa_soh": "0.920–0.945",
        },
        {
            "componente": "Especialista B",
            "modelo": criar_especialista(
                1
            ).__class__.__name__,
            "n_features": len(
                FEATURES_EXPERTS
            ),
            "faixa_soh": "0.940–0.975",
        },
        {
            "componente": "Especialista C",
            "modelo": criar_especialista(
                2
            ).__class__.__name__,
            "n_features": len(
                FEATURES_EXPERTS
            ),
            "faixa_soh": "0.970–1.005",
        },
        {
            "componente": "Gate",
            "modelo": criar_gate_xgboost(
            ).__class__.__name__,
            "n_features": len(
                FEATURES_GATE
            ),
            "faixa_soh": "Classes A/B/C",
        },
    ]
)

print("=" * 100)
print("CONFIGURAÇÃO DOS MODELOS FINAIS")
print("=" * 100)

display(model_audit)

print(
    "\nFolds externos:",
    N_SPLITS,
)

print(
    "Folds internos para os rótulos OOF do gate:",
    INNER_OOF_SPLITS,
)

print(
    "Fronteira A/B do router:",
    ROUTER_BOUNDARY_A_B,
)

print(
    "Fronteira B/C do router:",
    ROUTER_BOUNDARY_B_C,
)

# ----------------------------------------------------------
# 11. Salvar parâmetros
# ----------------------------------------------------------

MODEL_CONFIG_PATH = (
    CONFIG_DIR
    / "final_model_parameters.json"
)

model_config_json = {
    "random_state":
        MODEL_RANDOM_STATE,

    "outer_cv_splits":
        N_SPLITS,

    "inner_oof_splits":
        INNER_OOF_SPLITS,

    "global_model":
        GLOBAL_MODEL_PARAMS,

    "expert_a_xgboost":
        EXPERT_A_XGB_PARAMS,

    "expert_b_c_extratrees":
        EXPERT_BC_ET_PARAMS,

    "gate_xgboost":
        GATE_XGB_PARAMS,

    "router_boundaries": {
        "a_b": ROUTER_BOUNDARY_A_B,
        "b_c": ROUTER_BOUNDARY_B_C,
    },

    "expert_class_names":
        EXPERT_CLASS_NAMES,
}

with open(
    MODEL_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        model_config_json,
        file,
        ensure_ascii=False,
        indent=4,
    )

model_audit.to_excel(
    AUDIT_DIR
    / "final_model_configuration.xlsx",
    index=False,
)

print("\nConfiguração salva em:")
print(MODEL_CONFIG_PATH)

CONFIGURAÇÃO DOS MODELOS FINAIS


,componente,modelo,n_features,faixa_soh
0,Global,ExtraTreesRegressor,21,0.920–1.005
1,Especialista A,XGBRegressor,21,0.920–0.945
2,Especialista B,ExtraTreesRegressor,21,0.940–0.975
3,Especialista C,ExtraTreesRegressor,21,0.970–1.005
4,Gate,XGBClassifier,76,Classes A/B/C



Folds externos: 5
Folds internos para os rótulos OOF do gate: 5
Fronteira A/B do router: 0.9425
Fronteira B/C do router: 0.9725

Configuração salva em:
C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\CONFIG\final_model_parameters.json


## 10. Construção dos rótulos OOF do gate

O gate precisa aprender qual especialista tende a produzir o menor erro para cada evento.

Para evitar que o melhor especialista seja definido a partir de previsões realizadas sobre os próprios dados de treinamento, são geradas predições out-of-fold internas:

1. o treino balanceado do fold externo é dividido em cinco partes;
2. em cada rodada, os especialistas são treinados em quatro partes;
3. os especialistas predizem a parte interna restante;
4. cada evento recebe três predições produzidas por modelos que não utilizaram aquele evento no ajuste;
5. o especialista com menor erro absoluto se torna o rótulo Oracle do gate.

Esses rótulos são utilizados exclusivamente no treino do classificador XGBoost.

In [15]:
# ==========================================================
# 10. RÓTULOS OOF INTERNOS PARA O GATE
# ==========================================================

from sklearn.model_selection import StratifiedKFold

# ----------------------------------------------------------
# 1. Função principal
# ----------------------------------------------------------

def gerar_rotulos_gate_oof(
    train_balanced_fold,
    outer_fold_id,
    n_inner_splits=INNER_OOF_SPLITS,
    random_state=MODEL_RANDOM_STATE,
    verbose=True,
):
    """
    Gera predições OOF internas dos três especialistas e
    define o melhor especialista para cada evento do treino
    balanceado de um fold externo.

    Parâmetros
    ----------
    train_balanced_fold : pandas.DataFrame
        Treino real + SMOTER do fold externo.

    outer_fold_id : int
        Identificador do fold externo.

    n_inner_splits : int
        Número de divisões internas.

    random_state : int
        Semente-base.

    verbose : bool
        Controla a impressão do progresso.

    Retorna
    -------
    dict com:
        gate_train_dataset
        expert_oof_predictions
        oracle_classes
        oracle_predictions
        oracle_audit
        inner_fold_audit
        expert_inner_metrics
    """

    total_start = time.perf_counter()

    data = (
        train_balanced_fold
        .copy()
        .reset_index(drop=True)
    )

    n_events = len(data)

    # ------------------------------------------------------
    # 1.1. Auditorias iniciais
    # ------------------------------------------------------

    required_columns = list(
        dict.fromkeys(
            [
                "event_id",
                "soh_bin",
                "sample_origin",
                TARGET_COL,
            ]
            + FEATURES_EXPERTS
            + FEATURES_GATE
        )
    )

    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Fold externo {outer_fold_id}: colunas ausentes:\n"
            f"{missing_columns}"
        )

    if data["event_id"].duplicated().any():
        raise ValueError(
            f"Fold externo {outer_fold_id}: existem "
            "event_id duplicados no treino balanceado."
        )

    if data["soh_bin"].isna().any():
        raise ValueError(
            f"Fold externo {outer_fold_id}: existem eventos "
            "sem soh_bin."
        )

    model_features_union = list(
        dict.fromkeys(
            FEATURES_EXPERTS
            + FEATURES_GATE
        )
    )

    invalid_count = int(
        data[model_features_union]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .isna()
        .sum()
        .sum()
    )

    if invalid_count > 0:
        raise ValueError(
            f"Fold externo {outer_fold_id}: existem "
            f"{invalid_count} valores inválidos nas features."
        )

    # ------------------------------------------------------
    # 1.2. Verificar suporte para a CV interna
    # ------------------------------------------------------

    inner_bin_counts = (
        data["soh_bin"]
        .value_counts(sort=False)
    )

    insufficient_bins = inner_bin_counts.loc[
        inner_bin_counts < n_inner_splits
    ]

    if not insufficient_bins.empty:
        raise ValueError(
            f"Fold externo {outer_fold_id}: há bins com menos "
            f"de {n_inner_splits} eventos:\n"
            f"{insufficient_bins}"
        )

    # ------------------------------------------------------
    # 1.3. Criar folds internos
    # ------------------------------------------------------

    inner_splitter = StratifiedKFold(
        n_splits=n_inner_splits,
        shuffle=True,
        random_state=(
            int(random_state)
            + int(outer_fold_id) * 100
        ),
    )

    stratification_labels = (
        data["soh_bin"]
        .astype(str)
        .to_numpy()
    )

    inner_fold_assignment = np.full(
        n_events,
        fill_value=-1,
        dtype=int,
    )

    for inner_fold_id, (_, valid_indices) in enumerate(
        inner_splitter.split(
            X=np.zeros(
                n_events,
                dtype=np.uint8,
            ),
            y=stratification_labels,
        )
    ):

        inner_fold_assignment[
            valid_indices
        ] = inner_fold_id

    if np.any(inner_fold_assignment < 0):
        raise ValueError(
            f"Fold externo {outer_fold_id}: existem eventos "
            "sem fold interno atribuído."
        )

    data["inner_fold"] = (
        inner_fold_assignment
    )

    # ------------------------------------------------------
    # 1.4. Matrizes para armazenar as predições OOF
    # ------------------------------------------------------

    expert_oof_matrix = np.full(
        shape=(n_events, 3),
        fill_value=np.nan,
        dtype=float,
    )

    prediction_count = np.zeros(
        n_events,
        dtype=int,
    )

    inner_audit_rows = []
    expert_metric_rows = []

    # ------------------------------------------------------
    # 1.5. Rodar os folds internos
    # ------------------------------------------------------

    for inner_fold_id in range(
        n_inner_splits
    ):

        inner_start = time.perf_counter()

        inner_valid_mask = (
            data["inner_fold"]
            .eq(inner_fold_id)
        )

        inner_train = (
            data.loc[
                ~inner_valid_mask
            ]
            .copy()
            .reset_index(drop=True)
        )

        inner_valid = (
            data.loc[
                inner_valid_mask
            ]
            .copy()
        )

        inner_valid_indices = (
            inner_valid.index.to_numpy()
        )

        if verbose:

            print(
                f"Fold externo {outer_fold_id} | "
                f"interno {inner_fold_id} | "
                f"treino={len(inner_train)} | "
                f"validação={len(inner_valid)}"
            )

        shared_ids = set(
            inner_train["event_id"]
        ).intersection(
            set(
                inner_valid["event_id"]
            )
        )

        if shared_ids:
            raise ValueError(
                f"Fold externo {outer_fold_id}, interno "
                f"{inner_fold_id}: vazamento de event_id."
            )

        # --------------------------------------------------
        # Treinar cada especialista
        # --------------------------------------------------

        for class_id in range(3):

            expert_config = (
                EXPERT_CONFIG[class_id]
            )

            train_range_mask = (
                obter_mascara_especialista(
                    inner_train,
                    class_id,
                )
            )

            expert_train = (
                inner_train.loc[
                    train_range_mask
                ]
                .copy()
                .reset_index(drop=True)
            )

            if len(expert_train) == 0:
                raise ValueError(
                    f"Fold externo {outer_fold_id}, interno "
                    f"{inner_fold_id}, especialista "
                    f"{class_id}: treino vazio."
                )

            expert_model = criar_especialista(
                class_id=class_id,
                random_state=(
                    int(random_state)
                    + int(outer_fold_id) * 1000
                    + int(inner_fold_id) * 10
                    + int(class_id)
                ),
            )

            train_start = time.perf_counter()

            expert_model.fit(
                expert_train[
                    FEATURES_EXPERTS
                ],
                expert_train[
                    TARGET_COL
                ],
            )

            expert_train_time = (
                time.perf_counter()
                - train_start
            )

            prediction_start = (
                time.perf_counter()
            )

            expert_predictions = np.asarray(
                expert_model.predict(
                    inner_valid[
                        FEATURES_EXPERTS
                    ]
                ),
                dtype=float,
            )

            expert_prediction_time = (
                time.perf_counter()
                - prediction_start
            )

            if not np.isfinite(
                expert_predictions
            ).all():

                raise ValueError(
                    f"Fold externo {outer_fold_id}, interno "
                    f"{inner_fold_id}, especialista "
                    f"{class_id}: predições inválidas."
                )

            expert_oof_matrix[
                inner_valid_indices,
                class_id,
            ] = expert_predictions

            # Avaliação do especialista apenas em sua faixa
            valid_range_mask = (
                obter_mascara_especialista(
                    inner_valid,
                    class_id,
                )
                .to_numpy()
            )

            if valid_range_mask.sum() > 0:

                metrics_range = (
                    calcular_metricas_regressao(
                        inner_valid.loc[
                            valid_range_mask,
                            TARGET_COL,
                        ],
                        expert_predictions[
                            valid_range_mask
                        ],
                    )
                )

            else:

                metrics_range = {
                    "rmse": np.nan,
                    "mae": np.nan,
                    "r2": np.nan,
                    "mape_%": np.nan,
                    "bias": np.nan,
                    "erro_abs_p95": np.nan,
                    "erro_abs_max": np.nan,
                }

            expert_metric_rows.append(
                {
                    "outer_fold":
                        int(outer_fold_id),

                    "inner_fold":
                        int(inner_fold_id),

                    "class_id":
                        int(class_id),

                    "especialista":
                        expert_config["name"],

                    "n_train_expert":
                        len(expert_train),

                    "n_valid_total":
                        len(inner_valid),

                    "n_valid_range":
                        int(
                            valid_range_mask.sum()
                        ),

                    "tempo_treino_s":
                        expert_train_time,

                    "tempo_predicao_s":
                        expert_prediction_time,

                    **metrics_range,
                }
            )

        prediction_count[
            inner_valid_indices
        ] += 1

        inner_time = (
            time.perf_counter()
            - inner_start
        )

        inner_audit_rows.append(
            {
                "outer_fold":
                    int(outer_fold_id),

                "inner_fold":
                    int(inner_fold_id),

                "n_train":
                    len(inner_train),

                "n_valid":
                    len(inner_valid),

                "n_train_real":
                    int(
                        inner_train[
                            "sample_origin"
                        ]
                        .eq("real")
                        .sum()
                    ),

                "n_train_smoter":
                    int(
                        inner_train[
                            "sample_origin"
                        ]
                        .eq("smoter")
                        .sum()
                    ),

                "n_valid_real":
                    int(
                        inner_valid[
                            "sample_origin"
                        ]
                        .eq("real")
                        .sum()
                    ),

                "n_valid_smoter":
                    int(
                        inner_valid[
                            "sample_origin"
                        ]
                        .eq("smoter")
                        .sum()
                    ),

                "event_ids_shared":
                    len(shared_ids),

                "tempo_total_s":
                    inner_time,
            }
        )

    # ------------------------------------------------------
    # 1.6. Auditoria final das predições OOF
    # ------------------------------------------------------

    if not np.all(
        prediction_count == 1
    ):
        values, counts = np.unique(
            prediction_count,
            return_counts=True,
        )

        raise ValueError(
            f"Fold externo {outer_fold_id}: cada evento "
            "deveria receber exatamente uma previsão OOF. "
            f"Contagens encontradas: "
            f"{dict(zip(values, counts))}"
        )

    if not np.isfinite(
        expert_oof_matrix
    ).all():

        invalid_positions = np.argwhere(
            ~np.isfinite(
                expert_oof_matrix
            )
        )

        raise ValueError(
            f"Fold externo {outer_fold_id}: existem "
            "predições OOF ausentes ou inválidas. "
            f"Primeiras posições: "
            f"{invalid_positions[:10].tolist()}"
        )

    # ------------------------------------------------------
    # 1.7. Construir o Oracle OOF
    # ------------------------------------------------------

    y_train = (
        data[TARGET_COL]
        .to_numpy(dtype=float)
    )

    oracle_result = (
        calcular_oracle_especialistas(
            y_true=y_train,
            expert_prediction_matrix=(
                expert_oof_matrix
            ),
        )
    )

    oracle_classes = (
        oracle_result["classes"]
    )

    oracle_predictions = (
        oracle_result["predictions"]
    )

    absolute_errors = (
        oracle_result["absolute_errors"]
    )

    # ------------------------------------------------------
    # 1.8. Dataset final de treino do gate
    # ------------------------------------------------------

    gate_train_dataset = (
        data.copy()
    )

    gate_train_dataset[
        "best_expert_class"
    ] = oracle_classes

    gate_train_dataset[
        "best_expert"
    ] = pd.Series(
        oracle_classes
    ).map(
        EXPERT_CLASS_NAMES
    )

    gate_train_dataset[
        "pred_oof_expert_A"
    ] = expert_oof_matrix[:, 0]

    gate_train_dataset[
        "pred_oof_expert_B"
    ] = expert_oof_matrix[:, 1]

    gate_train_dataset[
        "pred_oof_expert_C"
    ] = expert_oof_matrix[:, 2]

    gate_train_dataset[
        "abs_error_oof_expert_A"
    ] = absolute_errors[:, 0]

    gate_train_dataset[
        "abs_error_oof_expert_B"
    ] = absolute_errors[:, 1]

    gate_train_dataset[
        "abs_error_oof_expert_C"
    ] = absolute_errors[:, 2]

    gate_train_dataset[
        "pred_oof_oracle"
    ] = oracle_predictions

    gate_train_dataset[
        "expert_margin"
    ] = oracle_result[
        "expert_margin"
    ]

    gate_train_dataset[
        "oracle_confidence"
    ] = oracle_result[
        "oracle_confidence"
    ]

    # ------------------------------------------------------
    # 1.9. Auditoria dos rótulos
    # ------------------------------------------------------

    oracle_audit = (
        gate_train_dataset[
            "best_expert"
        ]
        .value_counts()
        .rename_axis(
            "best_expert"
        )
        .reset_index(
            name="n_eventos"
        )
    )

    oracle_audit[
        "percentual_%"
    ] = (
        100
        * oracle_audit[
            "n_eventos"
        ]
        / len(
            gate_train_dataset
        )
    ).round(2)

    oracle_audit.insert(
        0,
        "outer_fold",
        int(outer_fold_id),
    )

    inner_fold_audit = pd.DataFrame(
        inner_audit_rows
    )

    expert_inner_metrics = pd.DataFrame(
        expert_metric_rows
    )

    total_time = (
        time.perf_counter()
        - total_start
    )

    if verbose:

        print(
            "\nDistribuição dos rótulos Oracle OOF:"
        )

        display(oracle_audit)

        print(
            "Tempo total dos rótulos OOF:",
            round(total_time / 60, 2),
            "min",
        )

    return {
        "gate_train_dataset":
            gate_train_dataset,

        "expert_oof_predictions":
            expert_oof_matrix,

        "oracle_classes":
            oracle_classes,

        "oracle_predictions":
            oracle_predictions,

        "oracle_audit":
            oracle_audit,

        "inner_fold_audit":
            inner_fold_audit,

        "expert_inner_metrics":
            expert_inner_metrics,

        "total_time_s":
            total_time,
    }


print(
    "Função gerar_rotulos_gate_oof criada com sucesso."
)

Função gerar_rotulos_gate_oof criada com sucesso.


## 11. Geração dos datasets OOF para treinamento do gate

Para cada fold externo, são geradas predições out-of-fold internas dos três especialistas.

O procedimento produz:

- uma previsão OOF de cada especialista para cada evento do treino balanceado;
- o erro absoluto OOF de cada especialista;
- o rótulo do especialista com menor erro;
- a margem entre o melhor e o segundo melhor especialista;
- a confiança relativa do Oracle;
- o dataset final utilizado para treinar o gate XGBoost.

Os resultados de cada fold são salvos separadamente. Caso a execução seja interrompida, os folds já concluídos podem ser reutilizados sem repetir os treinamentos.

In [16]:
# ==========================================================
# 11. GERAR OS DATASETS OOF PARA O GATE NOS CINCO FOLDS
# ==========================================================

import gc
import time
from pathlib import Path

# ----------------------------------------------------------
# 1. Configuração de execução
# ----------------------------------------------------------

# True:
# reutiliza resultados já salvos e válidos.
#
# False:
# refaz todos os treinamentos internos.
REUSE_GATE_OOF = True

GATE_OOF_DIR = OUTPUT_DIR / "gate_oof"

GATE_OOF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ----------------------------------------------------------
# 2. Estruturas consolidadas
# ----------------------------------------------------------

GATE_OOF_DATA = {}

gate_oof_summary_rows = []
oracle_audit_all = []
inner_fold_audit_all = []
expert_inner_metrics_all = []

gate_oof_total_start = time.perf_counter()

# ----------------------------------------------------------
# 3. Colunas mínimas esperadas no resultado salvo
# ----------------------------------------------------------

REQUIRED_GATE_OOF_COLUMNS = list(
    dict.fromkeys(
        [
            "event_id",
            "sample_origin",
            "inner_fold",
            TARGET_COL,
            "best_expert_class",
            "best_expert",
            "pred_oof_expert_A",
            "pred_oof_expert_B",
            "pred_oof_expert_C",
            "abs_error_oof_expert_A",
            "abs_error_oof_expert_B",
            "abs_error_oof_expert_C",
            "pred_oof_oracle",
            "expert_margin",
            "oracle_confidence",
        ]
        + FEATURES_GATE
    )
)

# ----------------------------------------------------------
# 4. Processar cada fold externo
# ----------------------------------------------------------

for outer_fold_id in range(N_SPLITS):

    fold_start = time.perf_counter()

    print("\n" + "=" * 110)
    print(
        f"RÓTULOS OOF DO GATE — FOLD EXTERNO "
        f"{outer_fold_id}"
    )
    print("=" * 110)

    fold_gate_dir = (
        GATE_OOF_DIR
        / f"fold_{outer_fold_id}"
    )

    fold_gate_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    gate_train_path = (
        fold_gate_dir
        / "gate_train_oof.csv"
    )

    oracle_audit_path = (
        fold_gate_dir
        / "oracle_oof_distribution.xlsx"
    )

    inner_audit_path = (
        fold_gate_dir
        / "inner_fold_audit.xlsx"
    )

    expert_metrics_path = (
        fold_gate_dir
        / "expert_inner_metrics.xlsx"
    )

    # ------------------------------------------------------
    # 4.1. Treino balanceado do fold externo
    # ------------------------------------------------------

    train_balanced_fold = (
        CV_FOLD_DATA[outer_fold_id][
            "train_balanced"
        ]
        .copy()
        .reset_index(drop=True)
    )

    n_expected = len(
        train_balanced_fold
    )

    loaded_from_disk = False

    # ------------------------------------------------------
    # 4.2. Tentar reutilizar resultado salvo
    # ------------------------------------------------------

    if (
        REUSE_GATE_OOF
        and gate_train_path.exists()
        and oracle_audit_path.exists()
        and inner_audit_path.exists()
        and expert_metrics_path.exists()
    ):

        print(
            "Resultado encontrado. "
            "Executando auditoria antes da reutilização..."
        )

        gate_train_dataset = pd.read_csv(
            gate_train_path
        )

        oracle_audit = pd.read_excel(
            oracle_audit_path
        )

        inner_fold_audit = pd.read_excel(
            inner_audit_path
        )

        expert_inner_metrics = pd.read_excel(
            expert_metrics_path
        )

        missing_saved_columns = [
            column
            for column in REQUIRED_GATE_OOF_COLUMNS
            if column not in gate_train_dataset.columns
        ]

        saved_is_valid = True

        if missing_saved_columns:

            print(
                "Arquivo salvo incompleto. "
                "Colunas ausentes:"
            )

            print(missing_saved_columns)

            saved_is_valid = False

        if len(gate_train_dataset) != n_expected:

            print(
                "Número de eventos salvo diferente do "
                "treino balanceado atual:"
            )

            print(
                "Salvo   :",
                len(gate_train_dataset)
            )

            print(
                "Esperado:",
                n_expected
            )

            saved_is_valid = False

        if (
            "event_id"
            in gate_train_dataset.columns
        ):

            ids_saved = set(
                gate_train_dataset[
                    "event_id"
                ].astype(str)
            )

            ids_current = set(
                train_balanced_fold[
                    "event_id"
                ].astype(str)
            )

            if ids_saved != ids_current:

                print(
                    "Os event_id salvos não correspondem "
                    "ao treino balanceado atual."
                )

                saved_is_valid = False

        if saved_is_valid:

            numeric_oof_columns = [
                "pred_oof_expert_A",
                "pred_oof_expert_B",
                "pred_oof_expert_C",
                "abs_error_oof_expert_A",
                "abs_error_oof_expert_B",
                "abs_error_oof_expert_C",
                "pred_oof_oracle",
                "expert_margin",
                "oracle_confidence",
            ]

            n_invalid_saved = int(
                gate_train_dataset[
                    numeric_oof_columns
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce",
                )
                .replace(
                    [np.inf, -np.inf],
                    np.nan,
                )
                .isna()
                .sum()
                .sum()
            )

            if n_invalid_saved > 0:

                print(
                    "O resultado salvo possui "
                    f"{n_invalid_saved} valores inválidos."
                )

                saved_is_valid = False

        if saved_is_valid:

            loaded_from_disk = True

            print(
                "Resultado salvo validado e reutilizado."
            )

    # ------------------------------------------------------
    # 4.3. Gerar resultado quando necessário
    # ------------------------------------------------------

    if not loaded_from_disk:

        print(
            "Gerando predições OOF internas..."
        )

        oof_result = gerar_rotulos_gate_oof(
            train_balanced_fold=(
                train_balanced_fold
            ),
            outer_fold_id=outer_fold_id,
            n_inner_splits=INNER_OOF_SPLITS,
            random_state=(
                MODEL_RANDOM_STATE
            ),
            verbose=True,
        )

        gate_train_dataset = (
            oof_result[
                "gate_train_dataset"
            ]
            .copy()
        )

        oracle_audit = (
            oof_result[
                "oracle_audit"
            ]
            .copy()
        )

        inner_fold_audit = (
            oof_result[
                "inner_fold_audit"
            ]
            .copy()
        )

        expert_inner_metrics = (
            oof_result[
                "expert_inner_metrics"
            ]
            .copy()
        )

        # --------------------------------------------------
        # Salvar imediatamente após concluir o fold
        # --------------------------------------------------

        gate_train_dataset.to_csv(
            gate_train_path,
            index=False,
        )

        oracle_audit.to_excel(
            oracle_audit_path,
            index=False,
        )

        inner_fold_audit.to_excel(
            inner_audit_path,
            index=False,
        )

        expert_inner_metrics.to_excel(
            expert_metrics_path,
            index=False,
        )

        print(
            "Resultados OOF salvos para o fold."
        )

    # ------------------------------------------------------
    # 4.4. Padronizar tipos após leitura ou geração
    # ------------------------------------------------------

    gate_train_dataset[
        "best_expert_class"
    ] = pd.to_numeric(
        gate_train_dataset[
            "best_expert_class"
        ],
        errors="raise",
    ).astype(int)

    gate_train_dataset[
        "inner_fold"
    ] = pd.to_numeric(
        gate_train_dataset[
            "inner_fold"
        ],
        errors="raise",
    ).astype(int)

    # ------------------------------------------------------
    # 4.5. Auditorias finais do fold
    # ------------------------------------------------------

    if len(gate_train_dataset) != n_expected:

        raise ValueError(
            f"Fold {outer_fold_id}: o dataset do gate "
            f"possui {len(gate_train_dataset)} eventos; "
            f"esperado={n_expected}."
        )

    if not gate_train_dataset[
        "event_id"
    ].is_unique:

        raise ValueError(
            f"Fold {outer_fold_id}: existem event_id "
            "duplicados no dataset OOF do gate."
        )

    if not np.isin(
        gate_train_dataset[
            "best_expert_class"
        ].to_numpy(),
        [0, 1, 2],
    ).all():

        raise ValueError(
            f"Fold {outer_fold_id}: existem classes "
            "Oracle inválidas."
        )

    inner_fold_values = set(
        gate_train_dataset[
            "inner_fold"
        ].unique()
        .tolist()
    )

    expected_inner_folds = set(
        range(INNER_OOF_SPLITS)
    )

    if (
        inner_fold_values
        != expected_inner_folds
    ):

        raise ValueError(
            f"Fold {outer_fold_id}: folds internos "
            f"encontrados={inner_fold_values}; "
            f"esperados={expected_inner_folds}."
        )

    # Conferência de alinhamento com o treino balanceado
    current_id_order = (
        train_balanced_fold[
            "event_id"
        ]
        .astype(str)
        .tolist()
    )

    gate_id_order = (
        gate_train_dataset[
            "event_id"
        ]
        .astype(str)
        .tolist()
    )

    if current_id_order != gate_id_order:

        # É permitido reordenar se os conjuntos forem iguais
        current_order_map = {
            event_id: position
            for position, event_id
            in enumerate(current_id_order)
        }

        gate_train_dataset[
            "_current_order"
        ] = (
            gate_train_dataset[
                "event_id"
            ]
            .astype(str)
            .map(current_order_map)
        )

        if gate_train_dataset[
            "_current_order"
        ].isna().any():

            raise ValueError(
                f"Fold {outer_fold_id}: falha ao "
                "realinhar o dataset OOF."
            )

        gate_train_dataset = (
            gate_train_dataset
            .sort_values(
                "_current_order"
            )
            .drop(
                columns="_current_order"
            )
            .reset_index(drop=True)
        )

    # Verificar que as features do gate são válidas
    n_invalid_gate_features = int(
        gate_train_dataset[
            FEATURES_GATE
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .isna()
        .sum()
        .sum()
    )

    if n_invalid_gate_features > 0:

        raise ValueError(
            f"Fold {outer_fold_id}: existem "
            f"{n_invalid_gate_features} valores inválidos "
            "nas features do gate."
        )

    # Verificar consistência do Oracle salvo
    expert_oof_matrix = (
        gate_train_dataset[
            [
                "pred_oof_expert_A",
                "pred_oof_expert_B",
                "pred_oof_expert_C",
            ]
        ]
        .to_numpy(dtype=float)
    )

    y_gate_oof = (
        gate_train_dataset[
            TARGET_COL
        ]
        .to_numpy(dtype=float)
    )

    oracle_check = (
        calcular_oracle_especialistas(
            y_true=y_gate_oof,
            expert_prediction_matrix=(
                expert_oof_matrix
            ),
        )
    )

    if not np.array_equal(
        oracle_check["classes"],
        gate_train_dataset[
            "best_expert_class"
        ].to_numpy(dtype=int),
    ):

        raise ValueError(
            f"Fold {outer_fold_id}: os rótulos Oracle "
            "salvos não correspondem às predições OOF."
        )

    # ------------------------------------------------------
    # 4.6. Recalcular distribuição para padronização
    # ------------------------------------------------------

    oracle_distribution = (
        gate_train_dataset[
            [
                "best_expert_class",
                "best_expert",
            ]
        ]
        .value_counts()
        .rename("n_eventos")
        .reset_index()
        .sort_values(
            "best_expert_class"
        )
        .reset_index(drop=True)
    )

    oracle_distribution[
        "percentual_%"
    ] = (
        100
        * oracle_distribution[
            "n_eventos"
        ]
        / len(
            gate_train_dataset
        )
    ).round(2)

    oracle_distribution.insert(
        0,
        "outer_fold",
        outer_fold_id,
    )

    # ------------------------------------------------------
    # 4.7. Métricas do Oracle OOF do treino
    # ------------------------------------------------------

    oracle_oof_metrics = (
        calcular_metricas_regressao(
            y_true=y_gate_oof,
            y_pred=oracle_check[
                "predictions"
            ],
        )
    )

    # ------------------------------------------------------
    # 4.8. Guardar em memória
    # ------------------------------------------------------

    GATE_OOF_DATA[
        outer_fold_id
    ] = {
        "gate_train_dataset":
            gate_train_dataset,

        "expert_oof_predictions":
            expert_oof_matrix,

        "oracle_classes":
            oracle_check["classes"],

        "oracle_predictions":
            oracle_check[
                "predictions"
            ],

        "oracle_distribution":
            oracle_distribution,

        "inner_fold_audit":
            inner_fold_audit,

        "expert_inner_metrics":
            expert_inner_metrics,
    }

    # Também conectar ao dicionário principal dos folds
    CV_FOLD_DATA[
        outer_fold_id
    ]["gate_train_oof"] = (
        gate_train_dataset
    )

    # ------------------------------------------------------
    # 4.9. Consolidar tabelas
    # ------------------------------------------------------

    oracle_audit_all.append(
        oracle_distribution
    )

    inner_fold_audit_all.append(
        inner_fold_audit
    )

    expert_inner_metrics_all.append(
        expert_inner_metrics
    )

    elapsed_fold = (
        time.perf_counter()
        - fold_start
    )

    class_counts = (
        gate_train_dataset[
            "best_expert_class"
        ]
        .value_counts()
        .to_dict()
    )

    gate_oof_summary_rows.append(
        {
            "outer_fold":
                outer_fold_id,

            "n_train_balanced":
                len(gate_train_dataset),

            "n_real":
                int(
                    gate_train_dataset[
                        "sample_origin"
                    ]
                    .eq("real")
                    .sum()
                ),

            "n_smoter":
                int(
                    gate_train_dataset[
                        "sample_origin"
                    ]
                    .eq("smoter")
                    .sum()
                ),

            "n_class_A":
                int(
                    class_counts.get(
                        0,
                        0,
                    )
                ),

            "n_class_B":
                int(
                    class_counts.get(
                        1,
                        0,
                    )
                ),

            "n_class_C":
                int(
                    class_counts.get(
                        2,
                        0,
                    )
                ),

            "percent_class_A_%":
                100
                * class_counts.get(0, 0)
                / len(gate_train_dataset),

            "percent_class_B_%":
                100
                * class_counts.get(1, 0)
                / len(gate_train_dataset),

            "percent_class_C_%":
                100
                * class_counts.get(2, 0)
                / len(gate_train_dataset),

            "oracle_oof_rmse":
                oracle_oof_metrics["rmse"],

            "oracle_oof_mae":
                oracle_oof_metrics["mae"],

            "oracle_oof_r2":
                oracle_oof_metrics["r2"],

            "expert_margin_mean":
                float(
                    gate_train_dataset[
                        "expert_margin"
                    ].mean()
                ),

            "oracle_confidence_mean":
                float(
                    gate_train_dataset[
                        "oracle_confidence"
                    ].mean()
                ),

            "reused_from_disk":
                loaded_from_disk,

            "tempo_total_fold_s":
                elapsed_fold,
        }
    )

    print("\nDistribuição Oracle OOF:")
    display(oracle_distribution)

    print(
        "Oracle OOF — RMSE:",
        f"{oracle_oof_metrics['rmse']:.6f}"
    )

    print(
        "Oracle OOF — MAE :",
        f"{oracle_oof_metrics['mae']:.6f}"
    )

    print(
        "Tempo do fold:",
        f"{elapsed_fold / 60:.2f} min"
    )

    gc.collect()

# ----------------------------------------------------------
# 5. Consolidar os cinco folds
# ----------------------------------------------------------

gate_oof_summary = pd.DataFrame(
    gate_oof_summary_rows
)

oracle_audit_cv = pd.concat(
    oracle_audit_all,
    axis=0,
    ignore_index=True,
)

inner_fold_audit_cv = pd.concat(
    inner_fold_audit_all,
    axis=0,
    ignore_index=True,
)

expert_inner_metrics_cv = pd.concat(
    expert_inner_metrics_all,
    axis=0,
    ignore_index=True,
)

gate_oof_total_time = (
    time.perf_counter()
    - gate_oof_total_start
)

# ----------------------------------------------------------
# 6. Auditorias consolidadas
# ----------------------------------------------------------

if len(GATE_OOF_DATA) != N_SPLITS:
    raise ValueError(
        "Nem todos os folds externos possuem "
        "dados OOF do gate."
    )

if (
    inner_fold_audit_cv[
        "event_ids_shared"
    ].sum()
    > 0
):

    raise ValueError(
        "Foi detectado vazamento em algum fold interno."
    )

# Cada fold interno deve aparecer uma vez por fold externo
inner_fold_counts = (
    inner_fold_audit_cv
    .groupby("outer_fold")[
        "inner_fold"
    ]
    .nunique()
)

if not inner_fold_counts.eq(
    INNER_OOF_SPLITS
).all():

    raise ValueError(
        "Algum fold externo não possui todos os "
        "folds internos esperados."
    )

# ----------------------------------------------------------
# 7. Salvar consolidados
# ----------------------------------------------------------

gate_oof_summary.to_excel(
    GATE_OOF_DIR
    / "gate_oof_cv_summary.xlsx",
    index=False,
)

oracle_audit_cv.to_excel(
    GATE_OOF_DIR
    / "oracle_oof_distribution_all_folds.xlsx",
    index=False,
)

inner_fold_audit_cv.to_excel(
    GATE_OOF_DIR
    / "inner_fold_audit_all_folds.xlsx",
    index=False,
)

expert_inner_metrics_cv.to_excel(
    GATE_OOF_DIR
    / "expert_inner_metrics_all_folds.xlsx",
    index=False,
)

# ----------------------------------------------------------
# 8. Exibir resumo final
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("RÓTULOS OOF DOS CINCO FOLDS CONCLUÍDOS")
print("=" * 110)

display(gate_oof_summary)

print(
    "\nTempo total:",
    round(
        gate_oof_total_time / 60,
        2,
    ),
    "min",
)

print("\nResultados disponíveis em:")
print("GATE_OOF_DATA[fold_id]")

print("\nArquivos salvos em:")
print(GATE_OOF_DIR)


RÓTULOS OOF DO GATE — FOLD EXTERNO 0
Resultado encontrado. Executando auditoria antes da reutilização...
Resultado salvo validado e reutilizado.

Distribuição Oracle OOF:


,outer_fold,best_expert_class,best_expert,n_eventos,percentual_%
0,0,0,A_XGBoost_Baixo,2072,17.21
1,0,1,B_ExtraTrees_Intermediario,3288,27.30
2,0,2,C_ExtraTrees_Alto,6682,55.49


Oracle OOF — RMSE: 0.004453
Oracle OOF — MAE : 0.003354
Tempo do fold: 0.02 min

RÓTULOS OOF DO GATE — FOLD EXTERNO 1
Resultado encontrado. Executando auditoria antes da reutilização...
Resultado salvo validado e reutilizado.

Distribuição Oracle OOF:


,outer_fold,best_expert_class,best_expert,n_eventos,percentual_%
0,1,0,A_XGBoost_Baixo,2084,17.31
1,1,1,B_ExtraTrees_Intermediario,3245,26.95
2,1,2,C_ExtraTrees_Alto,6712,55.74


Oracle OOF — RMSE: 0.004474
Oracle OOF — MAE : 0.003368
Tempo do fold: 0.02 min

RÓTULOS OOF DO GATE — FOLD EXTERNO 2
Resultado encontrado. Executando auditoria antes da reutilização...
Resultado salvo validado e reutilizado.

Distribuição Oracle OOF:


...

Oracle OOF — RMSE: 0.004410
Oracle OOF — MAE : 0.003326
Tempo do fold: 0.02 min

RÓTULOS OOF DO GATE — FOLD EXTERNO 4
Resultado encontrado. Executando auditoria antes da reutilização...
Resultado salvo validado e reutilizado.

Distribuição Oracle OOF:


,outer_fold,best_expert_class,best_expert,n_eventos,percentual_%
0,4,0,A_XGBoost_Baixo,2094,17.39
1,4,1,B_ExtraTrees_Intermediario,3233,26.85
2,4,2,C_ExtraTrees_Alto,6716,55.77


Oracle OOF — RMSE: 0.004422
Oracle OOF — MAE : 0.003324
Tempo do fold: 0.02 min

RÓTULOS OOF DOS CINCO FOLDS CONCLUÍDOS


,outer_fold,n_train_balanced,n_real,n_smoter,n_class_A,n_class_B,n_class_C,percent_class_A_%,percent_class_B_%,percent_class_C_%,oracle_oof_rmse,oracle_oof_mae,oracle_oof_r2,expert_margin_mean,oracle_confidence_mean,reused_from_disk,tempo_total_fold_s
0,0,12042,9207,2835,2072,3288,6682,17.206444,27.304434,55.489121,0.004453,0.003354,0.962888,0.016067,0.786593,True,1.158867
1,1,12041,9207,2834,2084,3245,6712,17.307533,26.949589,55.742878,0.004474,0.003368,0.962344,0.016143,0.788775,True,1.106797
2,2,12043,9207,2836,2067,3270,6706,17.163497,27.152703,55.683800,0.004449,0.003340,0.962736,0.016184,0.789657,True,1.092064
3,3,12043,9207,2836,2067,3250,6726,17.163497,26.986631,55.849871,0.004410,0.003326,0.963575,0.016232,0.789697,True,1.115077
4,4,12043,9208,2835,2094,3233,6716,17.387694,26.845470,55.766836,0.004422,0.003324,0.963267,0.016054,0.787149,True,1.065845



Tempo total: 0.1 min

Resultados disponíveis em:
GATE_OOF_DATA[fold_id]

Arquivos salvos em:
C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\OUTPUT\gate_oof


# 12. Treinamento dos modelos finais (5-fold Cross Validation)

Nesta etapa são treinados, para cada fold externo:

- Modelo Global ExtraTrees
- Especialista A (SOH baixo)
- Especialista B (SOH intermediário)
- Especialista C (SOH alto)
- Gate XGBoost

Todos os modelos utilizam exclusivamente o conjunto de treino correspondente ao fold externo.

O conjunto de validação permanece completamente isolado e é utilizado apenas para avaliação.

Os modelos treinados são armazenados para posterior inferência do MoE.

In [17]:
# ==========================================================
# 12. TREINAMENTO DOS MODELOS FINAIS NOS 5 FOLDS
# ==========================================================

import gc
import time
import joblib
import numpy as np
import pandas as pd

# ----------------------------------------------------------
# 1. Diretório dos modelos
# ----------------------------------------------------------

FINAL_MODELS_DIR = (
    OUTPUT_DIR
    / "trained_models"
)

FINAL_MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ----------------------------------------------------------
# 2. Estruturas de resultados
# ----------------------------------------------------------

FINAL_MODELS = {}

training_summary_rows = []
expert_training_summary_rows = []

training_total_start = time.perf_counter()

# ----------------------------------------------------------
# 3. Treinamento fold a fold
# ----------------------------------------------------------

for fold_id in range(N_SPLITS):

    fold_start = time.perf_counter()

    print("\n" + "=" * 100)
    print(f"TREINAMENTO FINAL — FOLD {fold_id}")
    print("=" * 100)

    # ------------------------------------------------------
    # 3.1. Recuperar os dados do fold
    # ------------------------------------------------------

    fold_data = CV_FOLD_DATA[fold_id]

    train_balanced_fold = (
        fold_data["train_balanced"]
        .copy()
        .reset_index(drop=True)
    )

    validation_real_fold = (
        fold_data["validation_real"]
        .copy()
        .reset_index(drop=True)
    )

    gate_train_oof = (
        fold_data["gate_train_oof"]
        .copy()
        .reset_index(drop=True)
    )

    print("Treino balanceado:", len(train_balanced_fold))
    print("Validação real   :", len(validation_real_fold))
    print("Treino do gate   :", len(gate_train_oof))

    # ------------------------------------------------------
    # 3.2. Auditorias antes do treinamento
    # ------------------------------------------------------

    shared_ids = set(
        train_balanced_fold["event_id"]
    ).intersection(
        set(
            validation_real_fold["event_id"]
        )
    )

    if shared_ids:
        raise ValueError(
            f"Fold {fold_id}: existem event_id compartilhados "
            "entre treino e validação."
        )

    if not validation_real_fold[
        "sample_origin"
    ].eq("real").all():

        raise ValueError(
            f"Fold {fold_id}: a validação contém "
            "eventos não reais."
        )

    if set(
        gate_train_oof["event_id"].astype(str)
    ) != set(
        train_balanced_fold["event_id"].astype(str)
    ):

        raise ValueError(
            f"Fold {fold_id}: o dataset OOF do gate não "
            "corresponde ao treino balanceado."
        )

    # ------------------------------------------------------
    # 3.3. Modelo global ExtraTrees
    # ------------------------------------------------------

    print("\nTreinando modelo global...")

    global_start = time.perf_counter()

    global_model = criar_modelo_global(
        random_state=(
            MODEL_RANDOM_STATE
            + fold_id
        )
    )

    global_model.fit(
        train_balanced_fold[
            FEATURES_GLOBAL
        ],
        train_balanced_fold[
            TARGET_COL
        ],
    )

    global_training_time = (
        time.perf_counter()
        - global_start
    )

    print(
        "Modelo global concluído:",
        round(global_training_time, 2),
        "s",
    )

    # ------------------------------------------------------
    # 3.4. Três especialistas
    # ------------------------------------------------------

    experts = {}
    expert_training_times = {}
    expert_training_sizes = {}

    print("\nTreinando especialistas...")

    experts_start = time.perf_counter()

    for class_id in range(3):

        expert_name = (
            EXPERT_CLASS_NAMES[
                class_id
            ]
        )

        expert_mask = (
            obter_mascara_especialista(
                train_balanced_fold,
                class_id,
            )
        )

        expert_train = (
            train_balanced_fold.loc[
                expert_mask
            ]
            .copy()
            .reset_index(drop=True)
        )

        if len(expert_train) == 0:
            raise ValueError(
                f"Fold {fold_id}: treino vazio para "
                f"o especialista {expert_name}."
            )

        expert_model = criar_especialista(
            class_id=class_id,
            random_state=(
                MODEL_RANDOM_STATE
                + fold_id * 10
                + class_id
            ),
        )

        expert_start = time.perf_counter()

        expert_model.fit(
            expert_train[
                FEATURES_EXPERTS
            ],
            expert_train[
                TARGET_COL
            ],
        )

        expert_time = (
            time.perf_counter()
            - expert_start
        )

        experts[class_id] = expert_model
        expert_training_times[class_id] = expert_time
        expert_training_sizes[class_id] = len(
            expert_train
        )

        expert_training_summary_rows.append(
            {
                "fold": fold_id,
                "class_id": class_id,
                "especialista": expert_name,
                "modelo": (
                    expert_model
                    .__class__
                    .__name__
                ),
                "soh_min": (
                    EXPERT_CONFIG[
                        class_id
                    ]["soh_min"]
                ),
                "soh_max": (
                    EXPERT_CONFIG[
                        class_id
                    ]["soh_max"]
                ),
                "n_train": len(
                    expert_train
                ),
                "n_train_real": int(
                    expert_train[
                        "sample_origin"
                    ]
                    .eq("real")
                    .sum()
                ),
                "n_train_smoter": int(
                    expert_train[
                        "sample_origin"
                    ]
                    .eq("smoter")
                    .sum()
                ),
                "n_features": len(
                    FEATURES_EXPERTS
                ),
                "tempo_treino_s": (
                    expert_time
                ),
            }
        )

        print(
            f"{expert_name}: "
            f"n={len(expert_train)} | "
            f"tempo={expert_time:.2f}s"
        )

    experts_training_time = (
        time.perf_counter()
        - experts_start
    )

    # ------------------------------------------------------
    # 3.5. Gate XGBoost
    # ------------------------------------------------------

    print("\nTreinando gate XGBoost...")

    X_gate_train = (
        gate_train_oof[
            FEATURES_GATE
        ]
    )

    y_gate_train = (
        gate_train_oof[
            "best_expert_class"
        ]
        .astype(int)
    )

    gate_classes = set(
        y_gate_train.unique()
        .tolist()
    )

    if gate_classes != {0, 1, 2}:
        raise ValueError(
            f"Fold {fold_id}: classes do gate "
            f"encontradas={gate_classes}; "
            "esperado={0, 1, 2}."
        )

    gate_start = time.perf_counter()

    gate_model = criar_gate_xgboost(
        random_state=(
            MODEL_RANDOM_STATE
            + fold_id
        )
    )

    gate_model.fit(
        X_gate_train,
        y_gate_train,
    )

    gate_training_time = (
        time.perf_counter()
        - gate_start
    )

    print(
        "Gate concluído:",
        round(gate_training_time, 2),
        "s",
    )

    # ------------------------------------------------------
    # 3.6. Salvar os modelos do fold
    # ------------------------------------------------------

    fold_model_dir = (
        FINAL_MODELS_DIR
        / f"fold_{fold_id}"
    )

    fold_model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    global_model_path = (
        fold_model_dir
        / "global_extratrees.joblib"
    )

    expert_a_path = (
        fold_model_dir
        / "expert_A_xgboost.joblib"
    )

    expert_b_path = (
        fold_model_dir
        / "expert_B_extratrees.joblib"
    )

    expert_c_path = (
        fold_model_dir
        / "expert_C_extratrees.joblib"
    )

    gate_model_path = (
        fold_model_dir
        / "gate_xgboost.joblib"
    )

    joblib.dump(
        global_model,
        global_model_path,
    )

    joblib.dump(
        experts[0],
        expert_a_path,
    )

    joblib.dump(
        experts[1],
        expert_b_path,
    )

    joblib.dump(
        experts[2],
        expert_c_path,
    )

    joblib.dump(
        gate_model,
        gate_model_path,
    )

    # ------------------------------------------------------
    # 3.7. Guardar em memória
    # ------------------------------------------------------

    FINAL_MODELS[fold_id] = {
        "global": global_model,
        "experts": experts,
        "gate": gate_model,
        "paths": {
            "global": global_model_path,
            "expert_A": expert_a_path,
            "expert_B": expert_b_path,
            "expert_C": expert_c_path,
            "gate": gate_model_path,
        },
    }

    CV_FOLD_DATA[
        fold_id
    ]["trained_models"] = (
        FINAL_MODELS[fold_id]
    )

    # ------------------------------------------------------
    # 3.8. Resumo do fold
    # ------------------------------------------------------

    fold_total_time = (
        time.perf_counter()
        - fold_start
    )

    gate_class_counts = (
        y_gate_train
        .value_counts()
        .to_dict()
    )

    training_summary_rows.append(
        {
            "fold": fold_id,

            "n_train_balanced":
                len(train_balanced_fold),

            "n_train_real":
                int(
                    train_balanced_fold[
                        "sample_origin"
                    ]
                    .eq("real")
                    .sum()
                ),

            "n_train_smoter":
                int(
                    train_balanced_fold[
                        "sample_origin"
                    ]
                    .eq("smoter")
                    .sum()
                ),

            "n_validation_real":
                len(
                    validation_real_fold
                ),

            "n_global_features":
                len(FEATURES_GLOBAL),

            "n_expert_features":
                len(FEATURES_EXPERTS),

            "n_gate_features":
                len(FEATURES_GATE),

            "n_expert_A":
                expert_training_sizes[0],

            "n_expert_B":
                expert_training_sizes[1],

            "n_expert_C":
                expert_training_sizes[2],

            "n_gate_class_A":
                int(
                    gate_class_counts.get(
                        0,
                        0,
                    )
                ),

            "n_gate_class_B":
                int(
                    gate_class_counts.get(
                        1,
                        0,
                    )
                ),

            "n_gate_class_C":
                int(
                    gate_class_counts.get(
                        2,
                        0,
                    )
                ),

            "tempo_global_s":
                global_training_time,

            "tempo_expert_A_s":
                expert_training_times[0],

            "tempo_expert_B_s":
                expert_training_times[1],

            "tempo_expert_C_s":
                expert_training_times[2],

            "tempo_experts_total_s":
                experts_training_time,

            "tempo_gate_s":
                gate_training_time,

            "tempo_total_fold_s":
                fold_total_time,
        }
    )

    print("\nResumo do fold:")
    print(
        "Tempo global:",
        round(global_training_time, 2),
        "s",
    )
    print(
        "Tempo especialistas:",
        round(experts_training_time, 2),
        "s",
    )
    print(
        "Tempo gate:",
        round(gate_training_time, 2),
        "s",
    )
    print(
        "Tempo total:",
        round(fold_total_time, 2),
        "s",
    )

    gc.collect()

# ----------------------------------------------------------
# 4. Consolidar resultados
# ----------------------------------------------------------

training_summary = pd.DataFrame(
    training_summary_rows
)

expert_training_summary = pd.DataFrame(
    expert_training_summary_rows
)

training_total_time = (
    time.perf_counter()
    - training_total_start
)

# ----------------------------------------------------------
# 5. Auditorias finais
# ----------------------------------------------------------

if len(FINAL_MODELS) != N_SPLITS:
    raise ValueError(
        "Nem todos os folds possuem modelos finais."
    )

for fold_id in range(N_SPLITS):

    required_model_keys = {
        "global",
        "experts",
        "gate",
        "paths",
    }

    found_model_keys = set(
        FINAL_MODELS[
            fold_id
        ].keys()
    )

    if not required_model_keys.issubset(
        found_model_keys
    ):
        raise ValueError(
            f"Fold {fold_id}: conjunto de modelos "
            "incompleto."
        )

    if set(
        FINAL_MODELS[
            fold_id
        ]["experts"].keys()
    ) != {0, 1, 2}:
        raise ValueError(
            f"Fold {fold_id}: especialistas incompletos."
        )

# ----------------------------------------------------------
# 6. Salvar resumos
# ----------------------------------------------------------

training_summary.to_excel(
    METRICS_DIR
    / "final_model_training_summary.xlsx",
    index=False,
)

expert_training_summary.to_excel(
    METRICS_DIR
    / "expert_training_summary.xlsx",
    index=False,
)

# ----------------------------------------------------------
# 7. Exibir
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("TREINAMENTO FINAL CONCLUÍDO")
print("=" * 110)

display(training_summary)

print("\nResumo dos especialistas:")
display(expert_training_summary)

print(
    "\nTempo total:",
    round(
        training_total_time / 60,
        2,
    ),
    "min",
)

print("\nModelos disponíveis em:")
print("FINAL_MODELS[fold_id]")

print("\nModelos salvos em:")
print(FINAL_MODELS_DIR)


TREINAMENTO FINAL — FOLD 0
Treino balanceado: 12042
Validação real   : 2302
Treino do gate   : 12042

Treinando modelo global...
Modelo global concluído: 26.77 s

Treinando especialistas...
A_XGBoost_Baixo: n=2300 | tempo=2.90s
B_ExtraTrees_Intermediario: n=3758 | tempo=6.73s
C_ExtraTrees_Alto: n=7211 | tempo=14.02s

Treinando gate XGBoost...
Gate concluído: 38.63 s

Resumo do fold:
Tempo global: 26.77 s
Tempo especialistas: 23.74 s
Tempo gate: 38.63 s
Tempo total: 100.2 s

TREINAMENTO FINAL — FOLD 1
Treino balanceado: 12041
Validação real   : 2302
Treino do gate   : 12041

Treinando modelo global...
Modelo global concluído: 29.39 s

Treinando especialistas...
A_XGBoost_Baixo: n=2300 | tempo=3.86s
B_ExtraTrees_Intermediario: n=3758 | tempo=8.28s
C_ExtraTrees_Alto: n=7210 | tempo=16.41s

Treinando gate XGBoost...
Gate concluído: 73.96 s

Resumo do fold:
Tempo global: 29.39 s
Tempo especialistas: 28.65 s
Tempo gate: 73.96 s
Tempo total: 149.59 s

TREINAMENTO FINAL — FOLD 2
Treino balanc

,fold,n_train_balanced,n_train_real,n_train_smoter,n_validation_real,n_global_features,n_expert_features,n_gate_features,n_expert_A,n_expert_B,...,n_gate_class_A,n_gate_class_B,n_gate_class_C,tempo_global_s,tempo_expert_A_s,tempo_expert_B_s,tempo_expert_C_s,tempo_experts_total_s,tempo_gate_s,tempo_total_fold_s
0,0,12042,9207,2835,2302,21,21,76,2300,3758,...,2072,3288,6682,26.767047,2.896501,6.734759,14.023873,23.738555,38.629390,100.201722
1,1,12041,9207,2834,2302,21,21,76,2300,3758,...,2084,3245,6712,29.386231,3.864624,8.275140,16.408559,28.652252,73.962409,149.591760
2,2,12043,9207,2836,2302,21,21,76,2300,3759,...,2067,3270,6706,40.231365,6.787908,16.288356,26.772527,50.033570,112.572754,227.565008
3,3,12043,9207,2836,2302,21,21,76,2300,3759,...,2067,3250,6726,41.482070,6.821205,15.056787,25.299301,47.367224,138.517254,259.823075
4,4,12043,9208,2835,2301,21,21,76,2300,3758,...,2094,3233,6716,48.776740,8.202657,17.196204,26.885704,52.467472,133.840225,267.103754



Resumo dos especialistas:


,fold,class_id,especialista,modelo,soh_min,soh_max,n_train,n_train_real,n_train_smoter,n_features,tempo_treino_s
0,0,0,A_XGBoost_Baixo,XGBRegressor,0.92,0.945,2300,500,1800,21,2.896501
1,0,1,B_ExtraTrees_Intermediario,ExtraTreesRegressor,0.94,0.975,3758,2660,1098,21,6.734759
2,0,2,C_ExtraTrees_Alto,ExtraTreesRegressor,0.97,1.005,7211,6954,257,21,14.023873
3,1,0,A_XGBoost_Baixo,XGBRegressor,0.92,0.945,2300,500,1800,21,3.864624
4,1,1,B_ExtraTrees_Intermediario,ExtraTreesRegressor,0.94,0.975,3758,2660,1098,21,8.275140
5,1,2,C_ExtraTrees_Alto,ExtraTreesRegressor,0.97,1.005,7210,6954,256,21,16.408559
6,2,0,A_XGBoost_Baixo,XGBRegressor,0.92,0.945,2300,500,1800,21,6.787908
7,2,1,B_ExtraTrees_Intermediario,ExtraTreesRegressor,0.94,0.975,3759,2660,1099,21,16.288356
8,2,2,C_ExtraTrees_Alto,ExtraTreesRegressor,0.97,1.005,7211,6954,257,21,26.772527
9,3,0,A_XGBoost_Baixo,XGBRegressor,0.92,0.945,2300,500,1800,21,6.821205



Tempo total: 16.77 min

Modelos disponíveis em:
FINAL_MODELS[fold_id]

Modelos salvos em:
C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\OUTPUT\trained_models


# 13. Inferência e avaliação nas validações externas

Os modelos treinados em cada fold são aplicados exclusivamente à validação externa correspondente, formada apenas por eventos reais.

São avaliadas quatro arquiteturas:

1. **ExtraTrees Global**  
   Predição direta do SOH usando as 20 features físicas e `mileage`.

2. **MoE com Gate XGBoost**  
   O gate seleciona um dos três especialistas para cada evento.

3. **MoE roteado pelo modelo global**  
   A previsão preliminar do ExtraTrees global determina o especialista:
   - A: previsão inferior a 0,9425;
   - B: previsão entre 0,9425 e 0,9725;
   - C: previsão igual ou superior a 0,9725.

4. **Oracle dos especialistas**  
   Seleciona retrospectivamente o especialista com menor erro absoluto. É utilizado apenas como limite teórico e não representa uma arquitetura aplicável em produção.

As predições dos cinco folds são posteriormente concatenadas, produzindo uma previsão out-of-fold para cada evento real do Dataset Gold.

In [ ]:
# ==========================================================
# 13. INFERÊNCIA NAS VALIDAÇÕES EXTERNAS
# ==========================================================

import gc
import time

import numpy as np
import pandas as pd

# ----------------------------------------------------------
# 1. Estruturas de resultados
# ----------------------------------------------------------

FINAL_PREDICTIONS = {}

fold_metric_rows = []
gate_metric_rows = []
expert_external_metric_rows = []
prediction_frames = []
routing_distribution_rows = []

inference_total_start = time.perf_counter()

ARCHITECTURE_PREDICTION_COLUMNS = {
    "ExtraTrees_Global_Mileage":
        "pred_global",

    "MoE_Gate_XGBoost_Experts_Mileage":
        "pred_moe_gate",

    "MoE_Router_ExtraTrees_Global":
        "pred_moe_global_router",

    "Oracle_Experts_Mileage":
        "pred_oracle",
}

# ----------------------------------------------------------
# 2. Inferência fold a fold
# ----------------------------------------------------------

for fold_id in range(N_SPLITS):

    fold_start = time.perf_counter()

    print("\n" + "=" * 110)
    print(f"INFERÊNCIA EXTERNA — FOLD {fold_id}")
    print("=" * 110)

    # ------------------------------------------------------
    # 2.1. Recuperar validação e modelos
    # ------------------------------------------------------

    validation_real_fold = (
        CV_FOLD_DATA[fold_id][
            "validation_real"
        ]
        .copy()
        .reset_index(drop=True)
    )

    fold_models = FINAL_MODELS[fold_id]

    global_model = (
        fold_models["global"]
    )

    expert_models = (
        fold_models["experts"]
    )

    gate_model = (
        fold_models["gate"]
    )

    n_validation = len(
        validation_real_fold
    )

    y_true = (
        validation_real_fold[
            TARGET_COL
        ]
        .to_numpy(dtype=float)
    )

    if not validation_real_fold[
        "sample_origin"
    ].eq("real").all():

        raise ValueError(
            f"Fold {fold_id}: a validação externa "
            "contém eventos sintéticos."
        )

    # ------------------------------------------------------
    # 2.2. Predição do modelo global
    # ------------------------------------------------------

    global_prediction_start = (
        time.perf_counter()
    )

    pred_global = np.asarray(
        global_model.predict(
            validation_real_fold[
                FEATURES_GLOBAL
            ]
        ),
        dtype=float,
    )

    global_prediction_time = (
        time.perf_counter()
        - global_prediction_start
    )

    # ------------------------------------------------------
    # 2.3. Predições dos três especialistas
    # ------------------------------------------------------

    expert_prediction_matrix = np.zeros(
        shape=(n_validation, 3),
        dtype=float,
    )

    expert_prediction_times = {}

    for class_id in range(3):

        prediction_start = (
            time.perf_counter()
        )

        expert_prediction_matrix[
            :,
            class_id,
        ] = np.asarray(
            expert_models[
                class_id
            ].predict(
                validation_real_fold[
                    FEATURES_EXPERTS
                ]
            ),
            dtype=float,
        )

        expert_prediction_times[
            class_id
        ] = (
            time.perf_counter()
            - prediction_start
        )

    # ------------------------------------------------------
    # 2.4. Gate XGBoost
    # ------------------------------------------------------

    gate_prediction_start = (
        time.perf_counter()
    )

    gate_pred_class = np.asarray(
        gate_model.predict(
            validation_real_fold[
                FEATURES_GATE
            ]
        ),
        dtype=int,
    )

    gate_pred_proba = np.asarray(
        gate_model.predict_proba(
            validation_real_fold[
                FEATURES_GATE
            ]
        ),
        dtype=float,
    )

    gate_prediction_time = (
        time.perf_counter()
        - gate_prediction_start
    )

    if gate_pred_proba.shape != (
        n_validation,
        3,
    ):
        raise ValueError(
            f"Fold {fold_id}: predict_proba do gate "
            f"possui shape {gate_pred_proba.shape}; "
            f"esperado=({n_validation}, 3)."
        )

    pred_moe_gate = (
        combinar_predicoes_especialistas(
            expert_prediction_matrix,
            gate_pred_class,
        )
    )

    # ------------------------------------------------------
    # 2.5. Roteamento pela previsão global
    # ------------------------------------------------------

    global_router_class = (
        rotear_pela_previsao_global(
            pred_global
        )
    )

    pred_moe_global_router = (
        combinar_predicoes_especialistas(
            expert_prediction_matrix,
            global_router_class,
        )
    )

    # ------------------------------------------------------
    # 2.6. Oracle externo real
    # ------------------------------------------------------

    oracle_result = (
        calcular_oracle_especialistas(
            y_true=y_true,
            expert_prediction_matrix=(
                expert_prediction_matrix
            ),
        )
    )

    oracle_class = (
        oracle_result["classes"]
    )

    pred_oracle = (
        oracle_result["predictions"]
    )

    # ------------------------------------------------------
    # 2.7. Auditoria numérica
    # ------------------------------------------------------

    prediction_vectors = {
        "pred_global":
            pred_global,

        "pred_expert_A":
            expert_prediction_matrix[:, 0],

        "pred_expert_B":
            expert_prediction_matrix[:, 1],

        "pred_expert_C":
            expert_prediction_matrix[:, 2],

        "pred_moe_gate":
            pred_moe_gate,

        "pred_moe_global_router":
            pred_moe_global_router,

        "pred_oracle":
            pred_oracle,
    }

    for name, values in (
        prediction_vectors.items()
    ):

        if len(values) != n_validation:
            raise ValueError(
                f"Fold {fold_id}: {name} possui "
                f"{len(values)} valores; "
                f"esperado={n_validation}."
            )

        if not np.isfinite(values).all():
            raise ValueError(
                f"Fold {fold_id}: {name} contém "
                "NaN ou Inf."
            )

    for name, classes in {
        "gate_pred_class":
            gate_pred_class,

        "global_router_class":
            global_router_class,

        "oracle_class":
            oracle_class,
    }.items():

        if not np.isin(
            classes,
            [0, 1, 2],
        ).all():

            raise ValueError(
                f"Fold {fold_id}: {name} possui "
                "classes inválidas."
            )

    # ------------------------------------------------------
    # 2.8. Métricas das quatro arquiteturas
    # ------------------------------------------------------

    architecture_predictions = {
        "ExtraTrees_Global_Mileage":
            pred_global,

        "MoE_Gate_XGBoost_Experts_Mileage":
            pred_moe_gate,

        "MoE_Router_ExtraTrees_Global":
            pred_moe_global_router,

        "Oracle_Experts_Mileage":
            pred_oracle,
    }

    for architecture_name, predictions in (
        architecture_predictions.items()
    ):

        architecture_metrics = (
            calcular_metricas_regressao(
                y_true=y_true,
                y_pred=predictions,
            )
        )

        fold_metric_rows.append(
            {
                "fold":
                    fold_id,

                "arquitetura":
                    architecture_name,

                "n_validation":
                    n_validation,

                **architecture_metrics,
            }
        )

    # ------------------------------------------------------
    # 2.9. Métricas dos especialistas individualmente
    # ------------------------------------------------------

    for class_id in range(3):

        expert_name = (
            EXPERT_CLASS_NAMES[
                class_id
            ]
        )

        expert_predictions = (
            expert_prediction_matrix[
                :,
                class_id
            ]
        )

        metrics_all_validation = (
            calcular_metricas_regressao(
                y_true=y_true,
                y_pred=expert_predictions,
            )
        )

        range_mask = (
            obter_mascara_especialista(
                validation_real_fold,
                class_id,
            )
            .to_numpy()
        )

        if range_mask.sum() > 0:

            metrics_expert_range = (
                calcular_metricas_regressao(
                    y_true=y_true[
                        range_mask
                    ],
                    y_pred=expert_predictions[
                        range_mask
                    ],
                )
            )

        else:

            metrics_expert_range = {
                "rmse": np.nan,
                "mae": np.nan,
                "r2": np.nan,
                "mape_%": np.nan,
                "bias": np.nan,
                "erro_abs_p95": np.nan,
                "erro_abs_max": np.nan,
            }

        expert_external_metric_rows.append(
            {
                "fold":
                    fold_id,

                "class_id":
                    class_id,

                "especialista":
                    expert_name,

                "n_validation_total":
                    n_validation,

                "n_validation_range":
                    int(
                        range_mask.sum()
                    ),

                "rmse_all":
                    metrics_all_validation[
                        "rmse"
                    ],

                "mae_all":
                    metrics_all_validation[
                        "mae"
                    ],

                "r2_all":
                    metrics_all_validation[
                        "r2"
                    ],

                "rmse_range":
                    metrics_expert_range[
                        "rmse"
                    ],

                "mae_range":
                    metrics_expert_range[
                        "mae"
                    ],

                "r2_range":
                    metrics_expert_range[
                        "r2"
                    ],

                "bias_range":
                    metrics_expert_range[
                        "bias"
                    ],

                "erro_abs_p95_range":
                    metrics_expert_range[
                        "erro_abs_p95"
                    ],

                "tempo_predicao_s":
                    expert_prediction_times[
                        class_id
                    ],
            }
        )

    # ------------------------------------------------------
    # 2.10. Avaliação do gate contra o Oracle externo
    # ------------------------------------------------------

    gate_metrics = (
        calcular_metricas_gate(
            y_true=oracle_class,
            y_pred=gate_pred_class,
            y_proba=gate_pred_proba,
        )
    )

    gate_metric_rows.append(
        {
            "fold":
                fold_id,

            "n_validation":
                n_validation,

            "n_oracle_A":
                int(
                    np.sum(
                        oracle_class == 0
                    )
                ),

            "n_oracle_B":
                int(
                    np.sum(
                        oracle_class == 1
                    )
                ),

            "n_oracle_C":
                int(
                    np.sum(
                        oracle_class == 2
                    )
                ),

            **gate_metrics,
        }
    )

    # ------------------------------------------------------
    # 2.11. Distribuição das rotas
    # ------------------------------------------------------

    for routing_name, classes in {
        "Gate_XGBoost":
            gate_pred_class,

        "Router_Global":
            global_router_class,

        "Oracle":
            oracle_class,
    }.items():

        class_counts = pd.Series(
            classes
        ).value_counts()

        for class_id in range(3):

            n_class = int(
                class_counts.get(
                    class_id,
                    0,
                )
            )

            routing_distribution_rows.append(
                {
                    "fold":
                        fold_id,

                    "roteamento":
                        routing_name,

                    "class_id":
                        class_id,

                    "especialista":
                        EXPERT_CLASS_NAMES[
                            class_id
                        ],

                    "n_eventos":
                        n_class,

                    "percentual_%":
                        100
                        * n_class
                        / n_validation,
                }
            )

    # ------------------------------------------------------
    # 2.12. DataFrame de predições por evento
    # ------------------------------------------------------

    prediction_columns_metadata = [
        column
        for column in [
            "event_id",
            "car",
            "mileage",
            "charge_segment",
            "soh_bin",
            "sample_origin",
            TARGET_COL,
        ]
        if column
        in validation_real_fold.columns
    ]

    fold_predictions_df = (
        validation_real_fold[
            prediction_columns_metadata
        ]
        .copy()
        .reset_index(drop=True)
    )

    fold_predictions_df.insert(
        0,
        "fold",
        fold_id,
    )

    fold_predictions_df = (
        fold_predictions_df.rename(
            columns={
                TARGET_COL: "soh_real"
            }
        )
    )

    fold_predictions_df[
        "pred_global"
    ] = pred_global

    fold_predictions_df[
        "pred_expert_A"
    ] = expert_prediction_matrix[:, 0]

    fold_predictions_df[
        "pred_expert_B"
    ] = expert_prediction_matrix[:, 1]

    fold_predictions_df[
        "pred_expert_C"
    ] = expert_prediction_matrix[:, 2]

    fold_predictions_df[
        "gate_pred_class"
    ] = gate_pred_class

    fold_predictions_df[
        "gate_pred_expert"
    ] = pd.Series(
        gate_pred_class
    ).map(
        EXPERT_CLASS_NAMES
    )

    fold_predictions_df[
        "gate_proba_A"
    ] = gate_pred_proba[:, 0]

    fold_predictions_df[
        "gate_proba_B"
    ] = gate_pred_proba[:, 1]

    fold_predictions_df[
        "gate_proba_C"
    ] = gate_pred_proba[:, 2]

    fold_predictions_df[
        "gate_max_probability"
    ] = np.max(
        gate_pred_proba,
        axis=1,
    )

    fold_predictions_df[
        "pred_moe_gate"
    ] = pred_moe_gate

    fold_predictions_df[
        "global_router_class"
    ] = global_router_class

    fold_predictions_df[
        "global_router_expert"
    ] = pd.Series(
        global_router_class
    ).map(
        EXPERT_CLASS_NAMES
    )

    fold_predictions_df[
        "pred_moe_global_router"
    ] = pred_moe_global_router

    fold_predictions_df[
        "oracle_class"
    ] = oracle_class

    fold_predictions_df[
        "oracle_expert"
    ] = pd.Series(
        oracle_class
    ).map(
        EXPERT_CLASS_NAMES
    )

    fold_predictions_df[
        "pred_oracle"
    ] = pred_oracle

    fold_predictions_df[
        "oracle_expert_margin"
    ] = oracle_result[
        "expert_margin"
    ]

    fold_predictions_df[
        "oracle_confidence"
    ] = oracle_result[
        "oracle_confidence"
    ]

    # Erros de cada arquitetura
    for architecture_name, pred_col in (
        ARCHITECTURE_PREDICTION_COLUMNS.items()
    ):

        error_col = (
            "error_"
            + architecture_name
        )

        absolute_error_col = (
            "abs_error_"
            + architecture_name
        )

        fold_predictions_df[
            error_col
        ] = (
            fold_predictions_df[
                pred_col
            ]
            - fold_predictions_df[
                "soh_real"
            ]
        )

        fold_predictions_df[
            absolute_error_col
        ] = np.abs(
            fold_predictions_df[
                error_col
            ]
        )

    # ------------------------------------------------------
    # 2.13. Salvar predições do fold
    # ------------------------------------------------------

    fold_prediction_dir = (
        PREDICTIONS_DIR
        / f"fold_{fold_id}"
    )

    fold_prediction_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    fold_predictions_df.to_csv(
        fold_prediction_dir
        / "validation_predictions.csv",
        index=False,
    )

    prediction_frames.append(
        fold_predictions_df
    )

    # ------------------------------------------------------
    # 2.14. Guardar em memória
    # ------------------------------------------------------

    FINAL_PREDICTIONS[fold_id] = {
        "y_true":
            y_true,

        "pred_global":
            pred_global,

        "expert_prediction_matrix":
            expert_prediction_matrix,

        "gate_pred_class":
            gate_pred_class,

        "gate_pred_proba":
            gate_pred_proba,

        "pred_moe_gate":
            pred_moe_gate,

        "global_router_class":
            global_router_class,

        "pred_moe_global_router":
            pred_moe_global_router,

        "oracle_class":
            oracle_class,

        "pred_oracle":
            pred_oracle,

        "predictions_df":
            fold_predictions_df,
    }

    CV_FOLD_DATA[
        fold_id
    ]["external_predictions"] = (
        FINAL_PREDICTIONS[fold_id]
    )

    # ------------------------------------------------------
    # 2.15. Exibir resumo do fold
    # ------------------------------------------------------

    fold_metrics_display = (
        pd.DataFrame(
            [
                row
                for row in fold_metric_rows
                if row["fold"] == fold_id
            ]
        )
        .sort_values(
            "rmse"
        )
        .reset_index(drop=True)
    )

    fold_elapsed = (
        time.perf_counter()
        - fold_start
    )

    print("\nMétricas do fold:")
    display(
        fold_metrics_display[
            [
                "arquitetura",
                "n_validation",
                "rmse",
                "mae",
                "r2",
                "bias",
                "erro_abs_p95",
                "erro_abs_max",
            ]
        ]
    )

    print(
        "Tempo de inferência do fold:",
        round(fold_elapsed, 3),
        "s",
    )

    gc.collect()

# ----------------------------------------------------------
# 3. Consolidar predições OOF externas
# ----------------------------------------------------------

oof_predictions = pd.concat(
    prediction_frames,
    axis=0,
    ignore_index=True,
)

oof_predictions = (
    oof_predictions
    .sort_values(
        [
            "fold",
            "event_id",
        ]
    )
    .reset_index(drop=True)
)

fold_metrics = pd.DataFrame(
    fold_metric_rows
)

gate_metrics_external = pd.DataFrame(
    gate_metric_rows
)

expert_external_metrics = pd.DataFrame(
    expert_external_metric_rows
)

routing_distribution = pd.DataFrame(
    routing_distribution_rows
)

inference_total_time = (
    time.perf_counter()
    - inference_total_start
)

# ----------------------------------------------------------
# 4. Auditorias OOF
# ----------------------------------------------------------

if len(oof_predictions) != len(df_cv):

    raise ValueError(
        "O número de predições OOF não coincide "
        "com o Dataset Gold limpo."
    )

if not oof_predictions[
    "event_id"
].is_unique:

    duplicated_oof_ids = (
        oof_predictions.loc[
            oof_predictions[
                "event_id"
            ].duplicated(
                keep=False
            ),
            "event_id",
        ]
        .tolist()
    )

    raise ValueError(
        "Alguns eventos aparecem mais de uma vez "
        f"nas predições OOF: {duplicated_oof_ids[:10]}"
    )

if set(
    oof_predictions["event_id"]
) != set(
    df_cv["event_id"]
):

    raise ValueError(
        "Os event_id das predições OOF não "
        "correspondem exatamente ao dataset."
    )

if not oof_predictions[
    "sample_origin"
].eq("real").all():

    raise ValueError(
        "As predições OOF externas contêm "
        "eventos sintéticos."
    )

# Cada evento deve ter sido previsto pelo fold
# ao qual foi originalmente atribuído.
fold_assignment_check = (
    oof_predictions[
        [
            "event_id",
            "fold",
        ]
    ]
    .merge(
        df_cv[
            [
                "event_id",
                "cv_fold",
            ]
        ],
        on="event_id",
        how="left",
        validate="one_to_one",
    )
)

if not fold_assignment_check[
    "fold"
].eq(
    fold_assignment_check[
        "cv_fold"
    ]
).all():

    raise ValueError(
        "Alguma predição OOF foi produzida pelo "
        "fold externo incorreto."
    )

# ----------------------------------------------------------
# 5. Salvar resultados
# ----------------------------------------------------------

oof_predictions.to_csv(
    PREDICTIONS_DIR
    / "oof_predictions_all_events.csv",
    index=False,
)

fold_metrics.to_excel(
    METRICS_DIR
    / "external_metrics_by_fold.xlsx",
    index=False,
)

gate_metrics_external.to_excel(
    METRICS_DIR
    / "gate_external_metrics_by_fold.xlsx",
    index=False,
)

expert_external_metrics.to_excel(
    METRICS_DIR
    / "expert_external_metrics_by_fold.xlsx",
    index=False,
)

routing_distribution.to_excel(
    METRICS_DIR
    / "routing_distribution_by_fold.xlsx",
    index=False,
)

# ----------------------------------------------------------
# 6. Exibir resumo
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("INFERÊNCIA EXTERNA CONCLUÍDA")
print("=" * 110)

print(
    "Eventos reais com previsão OOF:",
    len(oof_predictions),
)

print(
    "Event IDs únicos:",
    oof_predictions[
        "event_id"
    ].nunique(),
)

print(
    "Tempo total de inferência:",
    round(
        inference_total_time,
        3,
    ),
    "s",
)

print("\nMétricas por fold:")
display(
    fold_metrics.sort_values(
        [
            "fold",
            "rmse",
        ]
    )
)

print("\nMétricas externas do gate:")
display(
    gate_metrics_external
)

print("\nPredições OOF salvas em:")
print(
    PREDICTIONS_DIR
    / "oof_predictions_all_events.csv"
)


INFERÊNCIA EXTERNA — FOLD 0

Métricas do fold:


,arquitetura,n_validation,rmse,mae,r2,bias,erro_abs_p95,erro_abs_max
0,Oracle_Experts_Mileage,2302,0.004684,0.003624,0.921233,-0.000594,0.009376,0.021184
1,ExtraTrees_Global_Mileage,2302,0.008037,0.005712,0.768084,-0.000340,0.017595,0.042615
2,MoE_Gate_XGBoost_Experts_Mileage,2302,0.008375,0.005425,0.748128,0.000809,0.018665,0.044196
3,MoE_Router_ExtraTrees_Global,2302,0.008654,0.005727,0.731122,0.000665,0.019128,0.051195


Tempo de inferência do fold: 13.586 s

INFERÊNCIA EXTERNA — FOLD 1

Métricas do fold:


,arquitetura,n_validation,rmse,mae,r2,bias,erro_abs_p95,erro_abs_max
0,Oracle_Experts_Mileage,2302,0.004705,0.003633,0.921152,-0.000573,0.009416,0.018946
1,ExtraTrees_Global_Mileage,2302,0.007861,0.005593,0.779888,-0.000480,0.016520,0.054608
2,MoE_Gate_XGBoost_Experts_Mileage,2302,0.008110,0.005214,0.765773,0.000781,0.017101,0.061446
3,MoE_Router_ExtraTrees_Global,2302,0.008140,0.005394,0.764036,0.000538,0.017740,0.061446


Tempo de inferência do fold: 15.534 s

INFERÊNCIA EXTERNA — FOLD 2

Métricas do fold:


...

Tempo de inferência do fold: 2.161 s

INFERÊNCIA EXTERNA CONCLUÍDA
Eventos reais com previsão OOF: 11509
Event IDs únicos: 11509
Tempo total de inferência: 43.991 s

Métricas por fold:


,fold,arquitetura,n_validation,rmse,mae,r2,mape_%,bias,erro_abs_p95,erro_abs_max
3,0,Oracle_Experts_Mileage,2302,0.004684,0.003624,0.921233,0.370670,-0.000594,0.009376,0.021184
0,0,ExtraTrees_Global_Mileage,2302,0.008037,0.005712,0.768084,0.586162,-0.000340,0.017595,0.042615
1,0,MoE_Gate_XGBoost_Experts_Mileage,2302,0.008375,0.005425,0.748128,0.558219,0.000809,0.018665,0.044196
2,0,MoE_Router_ExtraTrees_Global,2302,0.008654,0.005727,0.731122,0.588820,0.000665,0.019128,0.051195
7,1,Oracle_Experts_Mileage,2302,0.004705,0.003633,0.921152,0.371551,-0.000573,0.009416,0.018946
4,1,ExtraTrees_Global_Mileage,2302,0.007861,0.005593,0.779888,0.573791,-0.000480,0.016520,0.054608
5,1,MoE_Gate_XGBoost_Experts_Mileage,2302,0.008110,0.005214,0.765773,0.536395,0.000781,0.017101,0.061446
6,1,MoE_Router_ExtraTrees_Global,2302,0.008140,0.005394,0.764036,0.554359,0.000538,0.017740,0.061446
11,2,Oracle_Experts_Mileage,2302,0.004806,0.003723,0.918158,0.380629,-0.000742,0.009861,0.016820
8,2,ExtraTrees_Global_Mileage,2302,0.008036,0.005763,0.771166,0.590776,-0.001054,0.016993,0.044107



Métricas externas do gate:


,fold,n_validation,n_oracle_A,n_oracle_B,n_oracle_C,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,log_loss
0,0,2302,130,563,1609,0.841008,0.673216,0.795312,0.673216,0.716204,0.833426,0.405676
1,1,2302,123,567,1612,0.853171,0.686169,0.788486,0.686169,0.725208,0.845749,0.397236
2,2,2302,126,556,1620,0.854474,0.693581,0.797074,0.693581,0.732500,0.848279,0.377668
3,3,2302,113,582,1607,0.847524,0.686483,0.767391,0.686483,0.719437,0.840212,0.388369
4,4,2301,130,560,1611,0.843112,0.653628,0.787071,0.653628,0.698035,0.832964,0.409426



Predições OOF salvas em:
C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\OUTPUT\predictions\oof_predictions_all_events.csv


# 15. Avaliação por faixas de SOH

As predições out-of-fold dos cinco folds externos são agrupadas em intervalos de SOH com largura de 0,005.

Para cada faixa são calculados:

- número de eventos reais;
- RMSE do modelo global;
- RMSE do MoE com gate XGBoost;
- RMSE do MoE roteado pela previsão global;
- RMSE do Oracle;
- ganho percentual das arquiteturas em relação ao modelo global.

O Oracle é mantido apenas como referência teórica. Portanto, também é identificado o melhor modelo aplicável em cada faixa, considerando somente:

- ExtraTrees global;
- MoE com gate XGBoost;
- MoE roteado pelo modelo global.

In [ ]:
# ==========================================================
# 15. RMSE CONSOLIDADO POR BIN DE SOH
# ==========================================================

import numpy as np
import pandas as pd

# ----------------------------------------------------------
# 1. Configuração das arquiteturas
# ----------------------------------------------------------

BIN_PREDICTION_COLUMNS = {
    "ExtraTrees_Global_Mileage":
        "pred_global",

    "MoE_Gate_XGBoost_Experts_Mileage":
        "pred_moe_gate",

    "MoE_Router_ExtraTrees_Global":
        "pred_moe_global_router",

    "Oracle_Experts_Mileage":
        "pred_oracle",
}

APPLICABLE_ARCHITECTURES = [
    "ExtraTrees_Global_Mileage",
    "MoE_Gate_XGBoost_Experts_Mileage",
    "MoE_Router_ExtraTrees_Global",
]

GLOBAL_ARCHITECTURE = (
    "ExtraTrees_Global_Mileage"
)

# ----------------------------------------------------------
# 2. Auditoria da tabela OOF
# ----------------------------------------------------------

if "oof_predictions" not in globals():
    raise NameError(
        "Execute primeiro a célula de inferência externa."
    )

required_bin_columns = [
    "event_id",
    "fold",
    "soh_real",
    "soh_bin",
    "sample_origin",
] + list(
    BIN_PREDICTION_COLUMNS.values()
)

missing_bin_columns = [
    column
    for column in required_bin_columns
    if column not in oof_predictions.columns
]

if missing_bin_columns:
    raise ValueError(
        "Colunas ausentes em oof_predictions:\n"
        f"{missing_bin_columns}"
    )

if len(oof_predictions) != len(df_cv):
    raise ValueError(
        "O número de eventos OOF não coincide com "
        "o Dataset Gold limpo."
    )

if not oof_predictions["event_id"].is_unique:
    raise ValueError(
        "Existem event_id duplicados nas predições OOF."
    )

if not oof_predictions[
    "sample_origin"
].eq("real").all():
    raise ValueError(
        "A avaliação por bin deve conter somente "
        "eventos reais."
    )

# ----------------------------------------------------------
# 3. Preparar DataFrame
# ----------------------------------------------------------

df_bin_evaluation = (
    oof_predictions
    .copy()
    .reset_index(drop=True)
)

# Ao carregar CSV, soh_bin pode ter voltado como texto.
# Recriamos os bins diretamente a partir do SOH real para
# garantir consistência.
df_bin_evaluation[
    "soh_bin_eval"
] = pd.cut(
    df_bin_evaluation[
        "soh_real"
    ],
    bins=SOH_BIN_EDGES,
    include_lowest=True,
    right=True,
)

if df_bin_evaluation[
    "soh_bin_eval"
].isna().any():

    invalid_rows = (
        df_bin_evaluation.loc[
            df_bin_evaluation[
                "soh_bin_eval"
            ].isna(),
            [
                "event_id",
                "soh_real",
            ],
        ]
    )

    display(invalid_rows.head())

    raise ValueError(
        "Alguns eventos ficaram sem bin de avaliação."
    )

# ----------------------------------------------------------
# 4. Função auxiliar de RMSE
# ----------------------------------------------------------

def rmse_group(
    group,
    prediction_column,
):
    return float(
        np.sqrt(
            mean_squared_error(
                group["soh_real"],
                group[prediction_column],
            )
        )
    )

# ----------------------------------------------------------
# 5. RMSE consolidado por bin
# ----------------------------------------------------------

bin_rows = []

for soh_bin_value, group in (
    df_bin_evaluation
    .groupby(
        "soh_bin_eval",
        observed=False,
        sort=True,
    )
):

    row = {
        "soh_bin":
            str(soh_bin_value),

        "soh_bin_left":
            float(
                soh_bin_value.left
            ),

        "soh_bin_right":
            float(
                soh_bin_value.right
            ),

        "n_eventos":
            len(group),

        "n_carros":
            (
                group["car"].nunique()
                if "car" in group.columns
                else np.nan
            ),

        "soh_real_mean":
            float(
                group["soh_real"].mean()
            ),

        "soh_real_std":
            float(
                group["soh_real"].std()
            ),

        "mileage_mean":
            (
                float(
                    group["mileage"].mean()
                )
                if "mileage" in group.columns
                else np.nan
            ),
    }

    for architecture_name, prediction_column in (
        BIN_PREDICTION_COLUMNS.items()
    ):

        row[
            f"RMSE_{architecture_name}"
        ] = rmse_group(
            group,
            prediction_column,
        )

    bin_rows.append(row)

rmse_by_soh_bin = pd.DataFrame(
    bin_rows
)

# ----------------------------------------------------------
# 6. Remover bins vazios, se existirem
# ----------------------------------------------------------

rmse_by_soh_bin = (
    rmse_by_soh_bin.loc[
        rmse_by_soh_bin[
            "n_eventos"
        ] > 0
    ]
    .copy()
    .reset_index(drop=True)
)

# ----------------------------------------------------------
# 7. Ganhos contra o ExtraTrees global
# ----------------------------------------------------------

global_rmse_column = (
    f"RMSE_{GLOBAL_ARCHITECTURE}"
)

for architecture_name in (
    BIN_PREDICTION_COLUMNS
):

    if (
        architecture_name
        == GLOBAL_ARCHITECTURE
    ):
        continue

    architecture_rmse_column = (
        f"RMSE_{architecture_name}"
    )

    gain_column = (
        f"ganho_{architecture_name}"
        "_vs_Global_%"
    )

    rmse_by_soh_bin[
        gain_column
    ] = (
        100
        * (
            rmse_by_soh_bin[
                global_rmse_column
            ]
            - rmse_by_soh_bin[
                architecture_rmse_column
            ]
        )
        / rmse_by_soh_bin[
            global_rmse_column
        ]
    )

# ----------------------------------------------------------
# 8. Melhor arquitetura aplicável por bin
# ----------------------------------------------------------

applicable_rmse_columns = {
    architecture_name:
        f"RMSE_{architecture_name}"

    for architecture_name
    in APPLICABLE_ARCHITECTURES
}

rmse_by_soh_bin[
    "melhor_modelo_aplicavel"
] = (
    rmse_by_soh_bin[
        list(
            applicable_rmse_columns.values()
        )
    ]
    .idxmin(axis=1)
    .str.replace(
        "RMSE_",
        "",
        regex=False,
    )
)

rmse_by_soh_bin[
    "melhor_rmse_aplicavel"
] = (
    rmse_by_soh_bin[
        list(
            applicable_rmse_columns.values()
        )
    ]
    .min(axis=1)
)

rmse_by_soh_bin[
    "melhor_modelo_incluindo_oracle"
] = (
    rmse_by_soh_bin[
        [
            f"RMSE_{name}"
            for name
            in BIN_PREDICTION_COLUMNS
        ]
    ]
    .idxmin(axis=1)
    .str.replace(
        "RMSE_",
        "",
        regex=False,
    )
)

rmse_by_soh_bin[
    "gap_melhor_aplicavel_para_oracle_%"
] = (
    100
    * (
        rmse_by_soh_bin[
            "melhor_rmse_aplicavel"
        ]
        - rmse_by_soh_bin[
            "RMSE_Oracle_Experts_Mileage"
        ]
    )
    / rmse_by_soh_bin[
        "melhor_rmse_aplicavel"
    ]
)

# ----------------------------------------------------------
# 9. Melhor arquitetura por evento, sem Oracle
# ----------------------------------------------------------

applicable_absolute_error_columns = {
    "ExtraTrees_Global_Mileage":
        "abs_error_ExtraTrees_Global_Mileage",

    "MoE_Gate_XGBoost_Experts_Mileage":
        "abs_error_MoE_Gate_XGBoost_Experts_Mileage",

    "MoE_Router_ExtraTrees_Global":
        "abs_error_MoE_Router_ExtraTrees_Global",
}

missing_error_columns = [
    column
    for column
    in applicable_absolute_error_columns.values()
    if column not in df_bin_evaluation.columns
]

if not missing_error_columns:

    df_bin_evaluation[
        "melhor_modelo_aplicavel_evento"
    ] = (
        df_bin_evaluation[
            list(
                applicable_absolute_error_columns.values()
            )
        ]
        .idxmin(axis=1)
        .str.replace(
            "abs_error_",
            "",
            regex=False,
        )
    )

    wins_by_bin = (
        df_bin_evaluation
        .groupby(
            [
                "soh_bin_eval",
                "melhor_modelo_aplicavel_evento",
            ],
            observed=True,
        )
        .size()
        .rename("n_vitorias")
        .reset_index()
    )

    total_by_bin = (
        df_bin_evaluation
        .groupby(
            "soh_bin_eval",
            observed=True,
        )
        .size()
        .rename("n_eventos_bin")
        .reset_index()
    )

    wins_by_bin = (
        wins_by_bin
        .merge(
            total_by_bin,
            on="soh_bin_eval",
            how="left",
            validate="many_to_one",
        )
    )

    wins_by_bin[
        "percentual_vitorias_%"
    ] = (
        100
        * wins_by_bin[
            "n_vitorias"
        ]
        / wins_by_bin[
            "n_eventos_bin"
        ]
    )

    wins_by_bin[
        "soh_bin"
    ] = wins_by_bin[
        "soh_bin_eval"
    ].astype(str)

    wins_by_bin = (
        wins_by_bin.drop(
            columns="soh_bin_eval"
        )
    )

else:

    wins_by_bin = pd.DataFrame()

# ----------------------------------------------------------
# 10. Resumo de bins vencidos
# ----------------------------------------------------------

bins_won_summary = (
    rmse_by_soh_bin[
        "melhor_modelo_aplicavel"
    ]
    .value_counts()
    .rename_axis("arquitetura")
    .reset_index(name="bins_vencidos")
)

bins_won_summary[
    "percentual_bins_%"
] = (
    100
    * bins_won_summary[
        "bins_vencidos"
    ]
    / len(rmse_by_soh_bin)
)

# ----------------------------------------------------------
# 11. Versão compacta para leitura
# ----------------------------------------------------------

rmse_by_soh_bin_compact = (
    rmse_by_soh_bin[
        [
            "soh_bin",
            "n_eventos",
            "n_carros",
            "RMSE_ExtraTrees_Global_Mileage",
            "RMSE_MoE_Gate_XGBoost_Experts_Mileage",
            "RMSE_MoE_Router_ExtraTrees_Global",
            "RMSE_Oracle_Experts_Mileage",
            (
                "ganho_MoE_Gate_XGBoost_Experts_Mileage"
                "_vs_Global_%"
            ),
            (
                "ganho_MoE_Router_ExtraTrees_Global"
                "_vs_Global_%"
            ),
            (
                "ganho_Oracle_Experts_Mileage"
                "_vs_Global_%"
            ),
            "melhor_modelo_aplicavel",
            "melhor_rmse_aplicavel",
            "gap_melhor_aplicavel_para_oracle_%",
        ]
    ]
    .copy()
)

# ----------------------------------------------------------
# 12. Salvar
# ----------------------------------------------------------

RMSE_BY_BIN_PATH = (
    METRICS_DIR
    / "rmse_by_soh_bin_oof.xlsx"
)

RMSE_BY_BIN_COMPACT_PATH = (
    METRICS_DIR
    / "rmse_by_soh_bin_oof_compact.xlsx"
)

rmse_by_soh_bin.to_excel(
    RMSE_BY_BIN_PATH,
    index=False,
)

rmse_by_soh_bin_compact.to_excel(
    RMSE_BY_BIN_COMPACT_PATH,
    index=False,
)

bins_won_summary.to_excel(
    METRICS_DIR
    / "bins_won_by_architecture.xlsx",
    index=False,
)

if not wins_by_bin.empty:

    wins_by_bin.to_excel(
        METRICS_DIR
        / "event_wins_by_bin.xlsx",
        index=False,
    )

# Adicionar ao workbook consolidado
RMSE_WORKBOOK_PATH = (
    METRICS_DIR
    / "rmse_by_bin_complete_workbook.xlsx"
)

with pd.ExcelWriter(
    RMSE_WORKBOOK_PATH,
    engine="openpyxl",
) as writer:

    rmse_by_soh_bin_compact.to_excel(
        writer,
        sheet_name="rmse_by_bin_compact",
        index=False,
    )

    rmse_by_soh_bin.to_excel(
        writer,
        sheet_name="rmse_by_bin_full",
        index=False,
    )

    bins_won_summary.to_excel(
        writer,
        sheet_name="bins_won",
        index=False,
    )

    if not wins_by_bin.empty:

        wins_by_bin.to_excel(
            writer,
            sheet_name="event_wins_by_bin",
            index=False,
        )

# ----------------------------------------------------------
# 13. Exibir resultados
# ----------------------------------------------------------

print("=" * 130)
print("RMSE OOF CONSOLIDADO POR BIN DE SOH")
print("=" * 130)

display(
    rmse_by_soh_bin_compact
)

print("\nNúmero de bins vencidos por arquitetura aplicável:")
display(
    bins_won_summary
)

if not wins_by_bin.empty:

    print(
        "\nVitórias por evento e por bin:"
    )

    display(
        wins_by_bin
        .sort_values(
            [
                "soh_bin",
                "percentual_vitorias_%",
            ],
            ascending=[
                True,
                False,
            ],
        )
    )

print("\nArquivos salvos:")
print("-", RMSE_BY_BIN_PATH)
print("-", RMSE_BY_BIN_COMPACT_PATH)
print("-", RMSE_WORKBOOK_PATH)

RMSE OOF CONSOLIDADO POR BIN DE SOH


,soh_bin,n_eventos,n_carros,RMSE_ExtraTrees_Global_Mileage,RMSE_MoE_Gate_XGBoost_Experts_Mileage,RMSE_MoE_Router_ExtraTrees_Global,RMSE_Oracle_Experts_Mileage,ganho_MoE_Gate_XGBoost_Experts_Mileage_vs_Global_%,ganho_MoE_Router_ExtraTrees_Global_vs_Global_%,ganho_Oracle_Experts_Mileage_vs_Global_%,melhor_modelo_aplicavel,melhor_rmse_aplicavel,gap_melhor_aplicavel_para_oracle_%
0,"(0.919, 0.925]",44,15,0.018049,0.021585,0.017520,0.009437,-19.587502,2.930027,47.713429,MoE_Router_ExtraTrees_Global,0.017520,46.135176
1,"(0.925, 0.93]",92,26,0.017135,0.016991,0.018553,0.007150,0.841968,-8.276548,58.269965,MoE_Gate_XGBoost_Experts_Mileage,0.016991,57.915628
2,"(0.93, 0.935]",118,28,0.014418,0.015905,0.016016,0.004120,-10.309256,-11.082108,71.425251,ExtraTrees_Global_Mileage,0.014418,71.425251
3,"(0.935, 0.94]",146,36,0.012176,0.016052,0.014365,0.003472,-31.829387,-17.976727,71.481354,ExtraTrees_Global_Mileage,0.012176,71.481354
4,"(0.94, 0.945]",225,45,0.012478,0.015865,0.013028,0.005727,-27.145840,-4.407223,54.106250,ExtraTrees_Global_Mileage,0.012478,54.106250
5,"(0.945, 0.95]",250,52,0.011286,0.013673,0.011853,0.006312,-21.149742,-5.030664,44.074847,ExtraTrees_Global_Mileage,0.011286,44.074847
6,"(0.95, 0.955]",328,55,0.011193,0.011603,0.011281,0.005619,-3.662240,-0.790172,49.798891,ExtraTrees_Global_Mileage,0.011193,49.798891
7,"(0.955, 0.96]",406,63,0.011077,0.011907,0.011968,0.004646,-7.491499,-8.037014,58.055258,ExtraTrees_Global_Mileage,0.011077,58.055258
8,"(0.96, 0.965]",543,70,0.009843,0.010931,0.011379,0.003900,-11.046262,-15.602894,60.377883,ExtraTrees_Global_Mileage,0.009843,60.377883
9,"(0.965, 0.97]",664,75,0.009142,0.010860,0.011275,0.004597,-18.797992,-23.333824,49.710970,ExtraTrees_Global_Mileage,0.009142,49.710970



Número de bins vencidos por arquitetura aplicável:


,arquitetura,bins_vencidos,percentual_bins_%
0,ExtraTrees_Global_Mileage,10,58.823529
1,MoE_Gate_XGBoost_Experts_Mileage,6,35.294118
2,MoE_Router_ExtraTrees_Global,1,5.882353



Vitórias por evento e por bin:


,melhor_modelo_aplicavel_evento,n_vitorias,n_eventos_bin,percentual_vitorias_%,soh_bin
1,MoE_Gate_XGBoost_Experts_Mileage,25,44,56.818182,"(0.919, 0.925]"
0,ExtraTrees_Global_Mileage,10,44,22.727273,"(0.919, 0.925]"
2,MoE_Router_ExtraTrees_Global,9,44,20.454545,"(0.919, 0.925]"
4,MoE_Gate_XGBoost_Experts_Mileage,64,92,69.565217,"(0.925, 0.93]"
3,ExtraTrees_Global_Mileage,19,92,20.652174,"(0.925, 0.93]"
5,MoE_Router_ExtraTrees_Global,9,92,9.782609,"(0.925, 0.93]"
7,MoE_Gate_XGBoost_Experts_Mileage,54,118,45.762712,"(0.93, 0.935]"
6,ExtraTrees_Global_Mileage,51,118,43.220339,"(0.93, 0.935]"
8,MoE_Router_ExtraTrees_Global,13,118,11.016949,"(0.93, 0.935]"
9,ExtraTrees_Global_Mileage,79,146,54.109589,"(0.935, 0.94]"



Arquivos salvos:
- C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\OUTPUT\metrics\rmse_by_soh_bin_oof.xlsx
- C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\OUTPUT\metrics\rmse_by_soh_bin_oof_compact.xlsx
- C:\SOH1\FASE1\battery_dataset_neurips23dataset_code\SOH_EV_Real_Data\OUTPUT\metrics\rmse_by_bin_complete_workbook.xlsx
